# Tests

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import config
from tqdm.auto import tqdm
from peft import PeftModel
import time
import json
import re
import json
import numpy as np
import pandas as pd
from IPython.display import display
from qwen_vl_utils import process_vision_info


# Import de tes modules personnalisés
from utils_data import load_reranking_data
from utils_reranking import RerankingCache, parse_vlm_ranking, calibrate_confidence_threshold
from utils_analysis import compute_autopsy, evaluate_and_save_results
from rerankers import Qwen2VLReranker

# ==========================================
# CONFIGURATION
# ==========================================
TOP_K_I2T = 10
TOP_K_T2I = 10
MODE_TEST = False 
NB_QUERIES_TEST = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Chargement du VLM (Modèle de base)
vlm = Qwen2VLReranker(model_id="Qwen/Qwen2-VL-7B-Instruct", device=device)
# ==========================================
# CHARGEMENT DES DONNÉES
# ==========================================
# 1. On charge la validation pour trouver le seuil optimal
val_dataset, val_texts , S_t2i_val_stage1, S_i2t_val_stage1, targets_t2i_val_gpu, targets_i2t_val_gpu = load_reranking_data("val", device)

# 2. On charge le set de test pour l'évaluation finale
dataset, all_texts, S_t2i_stage1, S_i2t_stage1, targets_t2i_gpu, targets_i2t_gpu = load_reranking_data("test", device)

limit_i2t = NB_QUERIES_TEST if MODE_TEST else len(targets_i2t_gpu)
limit_t2i = NB_QUERIES_TEST if MODE_TEST else len(targets_t2i_gpu)

/home/marion/MIRAGE_TFE/env_marion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Chargement de Qwen/Qwen2-VL-7B-Instruct en Bfloat16 (Sans quantification)...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 730/730 [00:02<00:00, 278.78it/s, Materializing param=model.visual.patch_embed.proj.weight]                          
Some parameters are on the meta device because they were offloaded to the cpu.


✅ Qwen2-VL chargé avec succès !

📦 Chargement des données pour le split : [VAL]...
  Chargé : ./data/flickr30k/index_sauvegardes/val_blip_img_index.bin (1014 vecteurs)
  Chargé : ./data/flickr30k/index_sauvegardes/val_blip_txt_index.bin (5070 vecteurs)
🔗 Calcul des matrices de similarité fusionnées...
✅ Données prêtes !

📦 Chargement des données pour le split : [TEST]...
  Chargé : ./data/flickr30k/index_sauvegardes/test_blip_img_index.bin (1000 vecteurs)
  Chargé : ./data/flickr30k/index_sauvegardes/test_blip_txt_index.bin (5000 vecteurs)
🔗 Calcul des matrices de similarité fusionnées...
✅ Données prêtes !


## I2T

In [2]:
'''
# ## Reranking I2T (SELF-CRITIQUE SOTA - 2ème Passe)

# %%
print("\n" + "="*50)
print("🚀 DÉMARRAGE I2T (SELF-CRITIQUE - 2ème Passe)")
print("="*50)

# 1. Chargement de tes prédictions initiales du VLM (Le fameux 0.971)
if not os.path.exists(config.CACHE_I2T_FILE):
    raise FileNotFoundError("Le cache initial est introuvable ! Lancez la première passe d'abord.")
    
with open(config.CACHE_I2T_FILE, 'r') as f:
    initial_vlm_predictions = json.load(f)

# 2. Création d'un NOUVEAU cache pour la critique (pour ne rien écraser)
CACHE_CRITIQUE_FILE = config.CACHE_I2T_FILE.replace('.json', '_critique.json')
cache_critique = RerankingCache(CACHE_CRITIQUE_FILE)

sorted_idx_i2t_base = torch.argsort(S_i2t_stage1, dim=1, descending=True)
sorted_idx_i2t_final = sorted_idx_i2t_base.clone()

# Sécurité LoRA
if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    vlm.model.disable_adapters()

temps_vlm_reel = 0.0
appels_vlm_reels = 0

for q_idx in tqdm(range(limit_i2t), desc="Inférence I2T Self-Critique"):
    top_k_idx = sorted_idx_i2t_base[q_idx, :TOP_K_I2T].tolist()
    
    # On vérifie si la critique a déjà été faite
    cached = cache_critique.get(q_idx)
    if cached:
        sorted_idx_i2t_final[q_idx, :TOP_K_I2T] = torch.tensor(cached, device=device)
        continue
        
    # On récupère l'ordre initial que le VLM avait choisi lors de la passe 1
    ordre_initial = initial_vlm_predictions[str(q_idx)]
    choix_top_1 = ordre_initial[0]
    
    image_query = dataset[q_idx]['image']
    textes_candidats = "\n".join([f"ID {idx}: {all_texts[idx]}" for idx in top_k_idx])
    
    # 🕵️‍♂️ LE PROMPT DE SELF-CRITIQUE
    prompt = f"""You are an expert image-to-text alignment evaluator.
    Here is a query image and {TOP_K_I2T} candidate descriptions with their unique IDs:
    
    {textes_candidats}

    Previously, you analyzed this image and ranked ID {choix_top_1} as the absolute best match.
    
    Your task: CRITICALLY RE-EVALUATE your choice.
    Look at the image very closely. Does the description in ID {choix_top_1} contain ANY subtle hallucination? (e.g., mentioning a color, an action, an object, or a background detail that is NOT visually present).
    
    - If ID {choix_top_1} is 100% flawless, you MUST keep it at the top.
    - If ID {choix_top_1} contains even a tiny hallucination, you MUST demote it and promote the true perfect match to the top.
    
    Think step-by-step to justify if your previous choice was correct or flawed.
    After your analysis, output the final ranking.
    
    Format:
    Final Ranking: [ID1, ID2, ID3, ID4, ID5]"""
    
    t0_call = time.time()
    
    response_text = vlm.generate_response(prompt, [image_query])
    
    t1_call = time.time()
    temps_vlm_reel += (t1_call - t0_call)
    appels_vlm_reels += 1

    # Parsing
    new_order = parse_vlm_ranking(response_text, top_k_idx)
    
    # Fallback si le parsing échoue ou renvoie une liste vide, on garde le 1er choix du VLM
    if len(new_order) == 0:
        new_order = ordre_initial
        
    sorted_idx_i2t_final[q_idx, :TOP_K_I2T] = torch.tensor(new_order, device=device)
    cache_critique.set(q_idx, new_order)
    
    if appels_vlm_reels % 5 == 0: 
        cache_critique.save()

cache_critique.save()

temps_total_vlm_i2t = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# --- Bilan I2T ---
evaluate_and_save_results(
    sorted_idx_i2t_base[:limit_i2t], 
    sorted_idx_i2t_final[:limit_i2t], 
    targets_i2t_gpu[:limit_i2t], 
    is_i2t=True, 
    csv_path=config.metriques_i2t_csv.replace('.csv', '_critique.csv'), 
    md_path=config.metriques_i2t_md.replace('.md', '_critique.md'),
    temps_vlm=temps_total_vlm_i2t
)

_ = compute_autopsy(sorted_idx_i2t_base, sorted_idx_i2t_final, targets_i2t_gpu[:limit_i2t], is_i2t=True, mask_vlm_called=[True]*limit_i2t)
'''

'\n# ## Reranking I2T (SELF-CRITIQUE SOTA - 2ème Passe)\n\n# %%\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE I2T (SELF-CRITIQUE - 2ème Passe)")\nprint("="*50)\n\n# 1. Chargement de tes prédictions initiales du VLM (Le fameux 0.971)\nif not os.path.exists(config.CACHE_I2T_FILE):\n    raise FileNotFoundError("Le cache initial est introuvable ! Lancez la première passe d\'abord.")\n\nwith open(config.CACHE_I2T_FILE, \'r\') as f:\n    initial_vlm_predictions = json.load(f)\n\n# 2. Création d\'un NOUVEAU cache pour la critique (pour ne rien écraser)\nCACHE_CRITIQUE_FILE = config.CACHE_I2T_FILE.replace(\'.json\', \'_critique.json\')\ncache_critique = RerankingCache(CACHE_CRITIQUE_FILE)\n\nsorted_idx_i2t_base = torch.argsort(S_i2t_stage1, dim=1, descending=True)\nsorted_idx_i2t_final = sorted_idx_i2t_base.clone()\n\n# Sécurité LoRA\nif hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):\n    vlm.model.disable_adapters()\n\ntemps_vlm_reel = 0.0\nappels_

In [3]:
'''
print("\n" + "="*50)
print("🚀 DÉMARRAGE I2T (FULL LISTWISE ZERO-SHOT - SANS CASCADE)")
print("="*50)

#cache_i2t = RerankingCache(config.CACHE_I2T_FILE.replace('.json', '_512.json'))
cache_i2t = RerankingCache(config.CACHE_I2T_FILE)
sorted_idx_i2t_base = torch.argsort(S_i2t_stage1, dim=1, descending=True)
sorted_idx_i2t_final = sorted_idx_i2t_base.clone()

# Sécurité : on s'assure que le VLM n'utilise pas LoRA pour la tâche Zero-Shot
if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA pour test Listwise Zero-Shot...")
    vlm.model.disable_adapters()

temps_vlm_reel = 0.0
appels_vlm_reels = 0

# Pas de masque conditionnel ici car on envoie TOUTES les requêtes (Full Reranking)
mask_vlm_called_i2t = [True] * limit_i2t

for q_idx in tqdm(range(limit_i2t), desc="Inférence I2T Full Listwise"):
    top_k_idx = sorted_idx_i2t_base[q_idx, :TOP_K_I2T].tolist()
    
    # Vérification du cache
    cached = cache_i2t.get(q_idx)
    if cached:
        sorted_idx_i2t_final[q_idx, :TOP_K_I2T] = torch.tensor(cached, device=device)
        continue
        
    image_query = dataset[q_idx]['image']
    textes_candidats = "\n".join([f"ID {idx}: {all_texts[idx]}" for idx in top_k_idx])
    
    # 🥇 TON ANCIEN PROMPT OPTIMISÉ (Le secret du 0.975)
    prompt = f"""You are an expert image-to-text alignment evaluator.
    Here is a query image and {TOP_K_I2T} candidate descriptions with their unique IDs:
    
    {textes_candidats}

    Task: Rank ALL {TOP_K_I2T} IDs from the most accurate description to the least accurate.
    
    CRUCIAL RULES:
    1. Multiple descriptions can be correct! You MUST group ALL factually correct descriptions at the very top of your ranking.
    2. Penalize hallucination: If a text mentions something NOT present in the image (e.g., wrong color, wrong action, wrong object), it must be ranked lower.
    3. Reward factual accuracy: Even short or simple descriptions must be ranked highly if they are 100% visually correct.
    
    First, think step-by-step. Analyze the visual elements of the image and explicitly compare them to the claims made in each candidate description. Point out any hallucinations or exact matches.
    
    After your analysis, you MUST conclude with a single line containing exactly {TOP_K_I2T} IDs in the new order.
    Format:
    Final Ranking: [ID1, ID2, ID3, ID4, ID5]"""
    
    t0_call = time.time()
    
    # Génération
    response_text = vlm.generate_response(prompt, [image_query])
    
    t1_call = time.time()
    temps_vlm_reel += (t1_call - t0_call)
    appels_vlm_reels += 1

    # --- PARSING PROPRE ---
    new_order = parse_vlm_ranking(response_text, top_k_idx)
    
    sorted_idx_i2t_final[q_idx, :TOP_K_I2T] = torch.tensor(new_order, device=device)
    cache_i2t.set(q_idx, new_order)
    
    if appels_vlm_reels % 5 == 0: 
        cache_i2t.save()

cache_i2t.save()

# --- Calcul du temps projeté ---
temps_total_vlm_i2t = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# --- Bilan I2T (Métriques et Sauvegarde) ---
evaluate_and_save_results(
    sorted_idx_i2t_base[:limit_i2t], 
    sorted_idx_i2t_final[:limit_i2t], 
    targets_i2t_gpu[:limit_i2t], 
    is_i2t=True, 
    csv_path=config.metriques_i2t_csv, 
    md_path=config.metriques_i2t_md,
    temps_vlm=temps_total_vlm_i2t
)

# --- Autopsie I2T (Sauvetages/Sabotages) ---
_ = compute_autopsy(sorted_idx_i2t_base, sorted_idx_i2t_final, targets_i2t_gpu[:limit_i2t], is_i2t=True, mask_vlm_called=mask_vlm_called_i2t)
'''

'\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE I2T (FULL LISTWISE ZERO-SHOT - SANS CASCADE)")\nprint("="*50)\n\n#cache_i2t = RerankingCache(config.CACHE_I2T_FILE.replace(\'.json\', \'_512.json\'))\ncache_i2t = RerankingCache(config.CACHE_I2T_FILE)\nsorted_idx_i2t_base = torch.argsort(S_i2t_stage1, dim=1, descending=True)\nsorted_idx_i2t_final = sorted_idx_i2t_base.clone()\n\n# Sécurité : on s\'assure que le VLM n\'utilise pas LoRA pour la tâche Zero-Shot\nif hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):\n    print("⚙️ Désactivation des adaptateurs LoRA pour test Listwise Zero-Shot...")\n    vlm.model.disable_adapters()\n\ntemps_vlm_reel = 0.0\nappels_vlm_reels = 0\n\n# Pas de masque conditionnel ici car on envoie TOUTES les requêtes (Full Reranking)\nmask_vlm_called_i2t = [True] * limit_i2t\n\nfor q_idx in tqdm(range(limit_i2t), desc="Inférence I2T Full Listwise"):\n    top_k_idx = sorted_idx_i2t_base[q_idx, :TOP_K_I2T].tolist()\n\n    # V

In [4]:
# 2. CONFIGURATION DES DEUX PASSES
passes_extraction_i2t = [
    {
        "nom": "Validation",
        "limite": len(targets_i2t_val_gpu),
        "S_base": S_i2t_val_stage1,
        "dataset": val_dataset,
        "textes": val_texts,
        "fichier_out": os.path.join(config.RESULTS_DIR, "scores_bruts_i2t_val.json")
    },
    {
        "nom": "Test",
        "limite": limit_i2t,
        "S_base": S_i2t_stage1,
        "dataset": dataset,
        "textes": all_texts,
        "fichier_out": os.path.join(config.RESULTS_DIR, "scores_bruts_i2t_test.json") 
    }
]

# 3. BOUCLE D'EXTRACTION EXHAUSTIVE I2T
for passe in passes_extraction_i2t:
    print(f"\n" + "="*50)
    print(f"🚀 DÉMARRAGE EXTRACTION I2T EXHAUSTIVE : Set de {passe['nom']}")
    print("="*50)
    
    fichier_out = passe['fichier_out']
    S_base = passe['S_base']
    limite = passe['limite']
    data_images = passe['dataset']
    data_textes = passe['textes']
    
    # Recalcul du Top K de base pour le set en cours
    sorted_idx_base = torch.argsort(S_base, dim=1, descending=True)
    scores_all_queries_i2t = {}

    # Chargement du cache existant
    if os.path.exists(fichier_out):
        with open(fichier_out, 'r') as f:
            scores_all_queries_i2t = json.load(f)

    for q_idx in tqdm(range(limite), desc=f"Extraction I2T Listwise ({passe['nom']})"):
        str_qidx = str(q_idx)
        top_k_idx = sorted_idx_base[q_idx, :TOP_K_I2T].tolist()
        
        if str_qidx in scores_all_queries_i2t:
            continue

        # Préparation du prompt Listwise
        image_query = data_images[q_idx]['image']
        textes_candidats = "\n".join([f"ID {idx}: {data_textes[idx]}" for idx in top_k_idx])
        
        prompt = f"""You are an expert image-to-text alignment evaluator.
        Here is a query image and {TOP_K_I2T} candidate descriptions with their unique IDs:
        
        {textes_candidats}

        Task: Rank ALL {TOP_K_I2T} IDs from the most accurate description to the least accurate.
        
        CRUCIAL RULES:
        1. Multiple descriptions can be correct! You MUST group ALL factually correct descriptions at the very top of your ranking.
        2. Penalize hallucination: If a text mentions something NOT present in the image (e.g., wrong color, wrong action, wrong object), it must be ranked lower.
        3. Reward factual accuracy: Even short or simple descriptions must be ranked highly if they are 100% visually correct.
        
        First, think step-by-step. Analyze the visual elements of the image and explicitly compare them to the claims made in each candidate description. Point out any hallucinations or exact matches.
        
        After your analysis, you MUST conclude with a single line containing exactly {TOP_K_I2T} IDs in the new order.
        Format:
        Final Ranking: [ID_A, ID_B, ID_C, ...]"""
        
        # Inférence VLM
        response_text = vlm.generate_response(prompt, [image_query])
        
        # Parsing
        new_order = parse_vlm_ranking(response_text, top_k_idx)
        
        # 🎯 Conversion du classement en Pseudo-Scores
        scores_vlm = []
        for idx in top_k_idx:
            if idx in new_order:
                rank = new_order.index(idx)
                # Formule linéaire : 1er=1.0, 2e=0.9, ..., 10e=0.1
                score = 1.0 - (rank / len(top_k_idx)) 
            else:
                score = 0.0 # Pénalité si le VLM a oublié cet ID
            scores_vlm.append(score)

        # Récupération des scores MIRAGE bruts
        scores_mirage = [S_base[q_idx, idx].item() for idx in top_k_idx]
        
        scores_all_queries_i2t[str_qidx] = {
            "candidate_ids": top_k_idx,
            "vlm_scores": scores_vlm,
            "mirage_scores": scores_mirage
        }
        
        with open(fichier_out, 'w') as f:
            json.dump(scores_all_queries_i2t, f)

    # Sauvegarde finale pour ce set
    with open(fichier_out, 'w') as f:
        json.dump(scores_all_queries_i2t, f)

    print(f"✅ Extraction I2T totale terminée et sauvegardée dans : {fichier_out}")


🚀 DÉMARRAGE EXTRACTION I2T EXHAUSTIVE : Set de Validation


Extraction I2T Listwise (Validation): 100%|██████████| 1014/1014 [00:00<00:00, 73644.17it/s]

✅ Extraction I2T totale terminée et sauvegardée dans : ./data/flickr30k/results/scores_bruts_i2t_val.json

🚀 DÉMARRAGE EXTRACTION I2T EXHAUSTIVE : Set de Test


Extraction I2T Listwise (Test): 100%|██████████| 1000/1000 [00:00<00:00, 78547.96it/s]

✅ Extraction I2T totale terminée et sauvegardée dans : ./data/flickr30k/results/scores_bruts_i2t_test.json


In [5]:
def evaluate_fusion_i2t(scores_dict, targets_gpu, method="additive", alpha=0.5, rrf_k=60, top_k=10, cascade_threshold=None):
    """Calcule le R@1 en simulant dynamiquement le Top-K et la Cascade pour l'I2T."""
    correct_count = 0
    total = len(scores_dict)
    reranked_results = {}
    mask_vlm_called = {} 
    
    for q_idx_str, data in scores_dict.items():
        q_idx = int(q_idx_str)
        
        # 🎯 GESTION DES MULTIPLES TARGETS POUR L'I2T
        target_tensor = targets_gpu[q_idx]
        if target_tensor.dim() == 0:
            valid_targets = [target_tensor.item()]
        else:
            valid_targets = [int(t) for t in target_tensor.tolist() if t != -1]
        
        ids = data["candidate_ids"][:top_k]
        vlm = np.array(data["vlm_scores"][:top_k])
        mirage = np.array(data["mirage_scores"][:top_k])
        
        # 🛡️ --- LOGIQUE DE CASCADE --- 🛡️
        ecart_confiance = mirage[0] - mirage[1] if len(mirage) > 1 else 0
        
        if cascade_threshold is not None and ecart_confiance > cascade_threshold:
            new_top_k = ids
            mask_vlm_called[q_idx] = False
        else:
            mask_vlm_called[q_idx] = True
            mirage_norm = (mirage - mirage.min()) / (mirage.max() - mirage.min() + 1e-8)
            
            if method == "additive":
                final_scores = (alpha * vlm) + ((1 - alpha) * mirage_norm)
            elif method == "multiplicative":
                final_scores = vlm * mirage_norm
            elif method == "maximum":
                final_scores = np.maximum(vlm, mirage_norm)
            elif method == "concatenation":
                final_scores = (vlm * 1000.0) + mirage_norm
            elif method == "rrf":
                rank_vlm = np.argsort(np.argsort(-vlm)) + 1
                rank_mirage = np.argsort(np.argsort(-mirage_norm)) + 1
                final_scores = (1.0 / (rrf_k + rank_vlm)) + (1.0 / (rrf_k + rank_mirage))
            elif method == "harmonique":
                final_scores = 2 * (vlm * mirage_norm) / (vlm + mirage_norm + 1e-8)
            elif method == "geometrique":
                final_scores = np.sqrt(vlm * mirage_norm)
            elif method == "rank_penalty":
                rank_mirage = np.argsort(np.argsort(-mirage)) # 0 pour le 1er, 1 pour le 2ème...
                # On retire 5% du score VLM pour chaque place de retard dans MIRAGE
                final_scores = vlm - (0.05 * rank_mirage) 
            elif method == "vlm_seul": # N'oublie pas celle-ci pour prouver l'utilité de ta fusion !
                final_scores = vlm
                
            best_indices = np.argsort(-final_scores) 
            new_top_k = [ids[i] for i in best_indices]
            
        reranked_results[q_idx] = new_top_k
        
        # 🎯 VÉRIFICATION I2T : Le top 1 est-il dans la liste des descriptions valides ?
        if new_top_k[0] in valid_targets:
            correct_count += 1
            
    return correct_count / total, reranked_results, mask_vlm_called


# ==========================================
# 0. PRÉPARATION DES DONNÉES (VAL ET TEST)
# ==========================================
RAW_SCORES_I2T_VAL_FILE = os.path.join(config.RESULTS_DIR, "scores_bruts_i2t_val_bis.json")
RAW_SCORES_I2T_TEST_FILE = os.path.join(config.RESULTS_DIR, "scores_bruts_i2t_test_bis.json")

with open(RAW_SCORES_I2T_VAL_FILE, 'r') as f:
    scores_bruts_val = json.load(f)
with open(RAW_SCORES_I2T_TEST_FILE, 'r') as f:
    scores_bruts_test = json.load(f)

# On s'assure d'avoir la matrice de base I2T pour les deux
sorted_idx_i2t_val_base = torch.argsort(S_i2t_val_stage1, dim=1, descending=True)
sorted_idx_i2t_test_base = torch.argsort(S_i2t_stage1, dim=1, descending=True)

# ==========================================
# 1. PHASE 1 : GRID SEARCH SUR VALIDATION
# ==========================================
print("\n" + "="*70)
print("🧪 PHASE 1 : RECHERCHE DES HYPERPARAMÈTRES I2T (SET DE VALIDATION)")
print("="*70)

# Attention: comme ton VLM a été prompté pour classer le Top 5 ou 10, 
options_top_k = [5, 10] 
methodes_simples = ["vlm_seul", "multiplicative", "maximum", "concatenation", "rrf", "harmonique", "geometrique", "rank_penalty"]
print("🎯 Calibrage des seuils de cascade sur le set de validation I2T...")
options_cascade = {"Sans Cascade": None}
recalls_a_tester = [0.70, 0.80, 0.85, 0.90, 0.95, 0.99, 1.00]

for tr in recalls_a_tester:
    seuil = calibrate_confidence_threshold(S_i2t_val_stage1, targets_i2t_val_gpu, is_i2t=True, target_recall=tr)
    options_cascade[f"Avec (Recall={tr:.2f})"] = seuil

resultats_val = []
grand_gagnant_val = {"r1": 0, "nom": "", "params": {}}

# Exécution de toutes les combinaisons sur le VAL
for top_k in options_top_k:
    for nom_cascade, val_cascade in options_cascade.items():
        
        # 1. Tests des méthodes simples
        for methode in methodes_simples:
            r1, _, mask = evaluate_fusion_i2t(scores_bruts_val, targets_i2t_val_gpu, method=methode, top_k=top_k, cascade_threshold=val_cascade)
            req_sauvees = sum(not v for v in mask.values())
            
            resultats_val.append({
                "Profondeur": f"Top {top_k}",
                "Cascade": nom_cascade.split(" ")[0], 
                "Stratégie de Fusion": methode.capitalize(),
                "R@1 Val (%)": r1 * 100,
                "VLM Évités": req_sauvees
            })
            
            if r1 > grand_gagnant_val["r1"]:
                grand_gagnant_val = {"r1": r1, "nom": f"{methode.capitalize()}_Top{top_k}_{nom_cascade.split(' ')[0]}", 
                                     "params": {"method": methode, "alpha": 0.5, "top_k": top_k, "cascade_threshold": val_cascade}}

        # 2. Test Additif avec Grid Search Intégré
        best_add_r1, best_alpha = 0, 0
        best_add_mask = {}
        for a in np.linspace(0, 1, 21):
            r1, _, mask = evaluate_fusion_i2t(scores_bruts_val, targets_i2t_val_gpu, method="additive", alpha=a, top_k=top_k, cascade_threshold=val_cascade)
            if r1 > best_add_r1:
                best_add_r1 = r1
                best_alpha = a
                best_add_mask = mask
                
        req_sauvees_add = sum(not v for v in best_add_mask.values())
        resultats_val.append({
            "Profondeur": f"Top {top_k}",
            "Cascade": nom_cascade.split(" ")[0],
            "Stratégie de Fusion": f"Grid Search (Alpha={best_alpha:.2f})",
            "R@1 Val (%)": best_add_r1 * 100,
            "VLM Évités": req_sauvees_add
        })
        
        if best_add_r1 > grand_gagnant_val["r1"]:
            grand_gagnant_val = {"r1": best_add_r1, "nom": f"Grid_Search_A{best_alpha:.2f}_Top{top_k}_{nom_cascade.split(' ')[0]}", 
                                 "params": {"method": "additive", "alpha": best_alpha, "top_k": top_k, "cascade_threshold": val_cascade}}


# Affichage du classement Validation
df_val = pd.DataFrame(resultats_val).sort_values(by=["R@1 Val (%)", "VLM Évités"], ascending=[False, False]).reset_index(drop=True)
display(df_val) # On n'affiche que le Top 10 pour ne pas surcharger

print("\n" + "⭐ "*15)
print(f"LA MEILLEURE CONFIGURATION I2T (VAL) EST : {grand_gagnant_val['nom']}")
print(f"R@1 sur Validation : {grand_gagnant_val['r1']*100:.2f} %")
print("⭐ "*15 + "\n")


# ==========================================
# 2. PHASE 2 : APPLICATION SUR LE TEST
# ==========================================
print("🔒 Hyperparamètres verrouillés. Déploiement sur le Set de Test...")
print("="*70)

# On extrait les paramètres parfaits
p = grand_gagnant_val["params"]

# 🎯 Un seul appel à la fonction sur le set de Test !
test_r1, dict_test, mask_test = evaluate_fusion_i2t(
    scores_bruts_test, 
    targets_i2t_gpu, 
    method=p["method"], 
    alpha=p["alpha"], 
    top_k=p["top_k"], 
    cascade_threshold=p["cascade_threshold"]
)

print(f"\n🏆 PERFORMANCE FINALE RÉELLE I2T (R@1 TEST) : {test_r1*100:.2f} % 🏆")
print(f"⚡ VLM économisés sur le Test : {sum(not v for v in mask_test.values())} requêtes\n")


# ==========================================
# 3. RECONSTRUCTION ET AUTOPSIE DU GAGNANT
# ==========================================
print(f"⚙️ Sauvegarde et Autopsie I2T de la configuration finale ({grand_gagnant_val['nom']})...")

sorted_idx_i2t_final = sorted_idx_i2t_test_base.clone()
for q_idx_str, new_top_k in dict_test.items():
    q_idx = int(q_idx_str)
    ordre_final = new_top_k + sorted_idx_i2t_test_base[q_idx, len(new_top_k):].tolist()
    sorted_idx_i2t_final[q_idx, :len(ordre_final)] = torch.tensor(ordre_final, device=device)

mask_vlm_called_list = [False] * limit_i2t
for q_idx_str, was_called in mask_test.items():
    mask_vlm_called_list[int(q_idx_str)] = was_called

file_suffix = f"_{grand_gagnant_val['nom'].replace(' ', '_').replace('.', '').lower()}_strict_test"

evaluate_and_save_results(
    sorted_idx_i2t_test_base[:limit_i2t], 
    sorted_idx_i2t_final[:limit_i2t], 
    targets_i2t_gpu[:limit_i2t], 
    is_i2t=True,  # IMPORTANT
    csv_path=config.metriques_i2t_csv.replace('.csv', f'{file_suffix}.csv'), 
    md_path=config.metriques_i2t_md.replace('.md', f'{file_suffix}.md'),
    temps_vlm=0.0
)

_ = compute_autopsy(
    sorted_idx_i2t_test_base, 
    sorted_idx_i2t_final, 
    targets_i2t_gpu[:limit_i2t], 
    is_i2t=True,  # IMPORTANT
    mask_vlm_called=mask_vlm_called_list
)


🧪 PHASE 1 : RECHERCHE DES HYPERPARAMÈTRES I2T (SET DE VALIDATION)
🎯 Calibrage des seuils de cascade sur le set de validation I2T...


📊 Seuil calibré pour attraper 70.0% des erreurs de validation : 0.01313
📊 Seuil calibré pour attraper 80.0% des erreurs de validation : 0.01536
📊 Seuil calibré pour attraper 85.0% des erreurs de validation : 0.01596
📊 Seuil calibré pour attraper 90.0% des erreurs de validation : 0.01819
📊 Seuil calibré pour attraper 95.0% des erreurs de validation : 0.02200
📊 Seuil calibré pour attraper 99.0% des erreurs de validation : 0.02456
📊 Seuil calibré pour attraper 100.0% des erreurs de validation : 0.02505


,Profondeur,Cascade,Stratégie de Fusion,R@1 Val (%),VLM Évités
0,Top 5,Avec,Vlm_seul,97.238659,297
1,Top 5,Avec,Maximum,97.238659,297
2,Top 5,Avec,Concatenation,97.238659,297
3,Top 5,Avec,Grid Search (Alpha=0.90),97.238659,297
4,Top 5,Avec,Vlm_seul,97.238659,216
...,...,...,...,...,...
139,Top 10,Avec,Harmonique,96.252465,216
140,Top 10,Avec,Harmonique,96.252465,177
141,Top 10,Avec,Harmonique,96.252465,169
142,Top 10,Sans,Harmonique,96.252465,0



⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ 
LA MEILLEURE CONFIGURATION I2T (VAL) EST : Vlm_seul_Top5_Avec
R@1 sur Validation : 97.24 %
⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ 

🔒 Hyperparamètres verrouillés. Déploiement sur le Set de Test...

🏆 PERFORMANCE FINALE RÉELLE I2T (R@1 TEST) : 97.50 % 🏆
⚡ VLM économisés sur le Test : 272 requêtes

⚙️ Sauvegarde et Autopsie I2T de la configuration finale (Vlm_seul_Top5_Avec)...

COMPARAISON FINALE : LATE FUSION vs VLM (I2T)
                                      R@1   R@5  R@10   mAP  NDCG  Temps (s)
1. Avant (Stage 1 Fusion)           0.960 0.999 1.000 0.883 0.948      0.000
2. Après (MIRAGE + VLM (Zero-Shot)) 0.975 0.999 1.000 0.885 0.950      0.000

✅ Résultats globaux (avec temps) sauvegardés dans resultats_i2t_vlm_seul_top5_avec_strict_test.csv et .md

--- BILAN COMPTABLE I2T ---
Total requêtes analysées : 728
✅ Sauvetages (Faux -> Vrai) : 18
⚠️ Sabotages (Vrai -> Faux) : 3
❌ Doubles Échecs (Faux -> Faux) : 17
🆗 Confirmations (Vrai -> Vrai) : 690
📈 GAIN NET DE 

In [6]:
# ==========================================
# 📊 ANALYSE DÉTAILLÉE DU RECALL (R@1 à R@5)
# ==========================================
print("\n" + "="*50)
print("📊 DISTRIBUTION DES PERFORMANCES (R@1 à R@5) I2T")
print("="*50)

recalls = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
total_queries = len(dict_test)

for q_idx_str, new_top_k in dict_test.items():
    q_idx = int(q_idx_str)
    
    # Récupération sécurisée des targets (comme dans ta fonction)
    target_tensor = targets_i2t_gpu[q_idx]
    if target_tensor.dim() == 0:
        valid_targets = [target_tensor.item()]
    else:
        valid_targets = [int(t) for t in target_tensor.tolist() if t != -1]
        
    # Calcul des Hits pour chaque profondeur (de 1 à 5)
    for k in range(1, 6):
        # Si au moins l'une des descriptions valides est dans le Top K actuel
        if any(t in new_top_k[:k] for t in valid_targets):
            recalls[k] += 1

# Affichage des résultats
for k in range(1, 6):
    score = (recalls[k] / total_queries) * 100
    print(f"R@{k} : {score:.2f} %")
print("="*50 + "\n")


📊 DISTRIBUTION DES PERFORMANCES (R@1 à R@5) I2T
R@1 : 97.50 %
R@2 : 98.50 %
R@3 : 99.20 %
R@4 : 99.60 %
R@5 : 99.90 %



## T2I

### Pointwise Prompt Ensembling (Sans cascade)

In [7]:
'''
print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (POINTWISE SOTA + PROMPT ENSEMBLING)")
print("="*50)

# 1. Désactivation du LoRA
if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA pour test Pointwise Zero-Shot...")
    vlm.model.disable_adapters()
else:
    print("✅ Le modèle est bien en mode de base (Zero-Shot).")

# 2. Initialisation d'un nouveau cache pour l'Ensembling
CACHE_ENSEMBLE_FILE = config.CACHE_T2I_FILE.replace('.json', '_ensemble.json')
cache_t2i = RerankingCache(CACHE_ENSEMBLE_FILE)

sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0
ALPHA = 0.5 

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Prompt Ensembling"):
    top_5 = sorted_idx_t2i_base[q_idx, :TOP_K_T2I].tolist()
    
    # Vérification du cache
    cached = cache_t2i.get(q_idx)
    if cached:
        new_top_5 = cached
    else:
        images_pil = [dataset[idx]['image'] for idx in top_5]
        requete = all_texts[q_idx]
        
        # 📝 LES 3 PROMPTS SOTA
        prompt_1 = f"Does the following image exactly match this description: '{requete}'? Answer strictly with 'Yes' or 'No'."
        prompt_2 = f"Look closely at the fine details and the background. Is this a perfect visual representation of: '{requete}'? Answer strictly with 'Yes' or 'No'."
        prompt_3 = f"Is the description '{requete}' 100% factually correct for every element visible in this image? Answer strictly with 'Yes' or 'No'."
        
        t0_call = time.time()
        
        # On score les 5 images pour chaque prompt (3 appels successifs au batch)
        scores_p1 = vlm.score_image_pointwise_batch([prompt_1] * 5, images_pil)
        scores_p2 = vlm.score_image_pointwise_batch([prompt_2] * 5, images_pil)
        scores_p3 = vlm.score_image_pointwise_batch([prompt_3] * 5, images_pil)
        
        # On fait la moyenne des 3 probabilités VLM pour chaque image
        scores_vlm_moyens = [(s1 + s2 + s3) / 3.0 for s1, s2, s3 in zip(scores_p1, scores_p2, scores_p3)]
        
        t1_call = time.time()

        temps_vlm_reel += (t1_call - t0_call)
        appels_vlm_reels += 1 

        # --- LATE FUSION ---
        scores_mirage_bruts = [S_t2i_stage1[q_idx, idx].item() for idx in top_5]
        
        # Normalisation Min-Max
        max_s = max(scores_mirage_bruts)
        min_s = min(scores_mirage_bruts)
        if max_s > min_s:
            scores_mirage_norm = [(s - min_s) / (max_s - min_s) for s in scores_mirage_bruts]
        else:
            scores_mirage_norm = [1.0] * 5

        # Fusion mathématique (Alpha * VLM_Moyen + (1-Alpha) * MIRAGE)
        final_scores = []
        for i, idx in enumerate(top_5):
            score_mixte = ((1 - ALPHA) * scores_mirage_norm[i]) + (ALPHA * scores_vlm_moyens[i])
            final_scores.append((score_mixte, idx))
            
        # Nouveau tri
        final_scores.sort(key=lambda x: x[0], reverse=True)
        new_top_5 = [idx for score, idx in final_scores]
        
        # Mise en cache
        cache_t2i.set(q_idx, new_top_5)
        if appels_vlm_reels % 5 == 0: 
            cache_t2i.save()
            
    ordre_final = new_top_5 + top_5[TOP_K_T2I:] if len(top_5) > TOP_K_T2I else new_top_5 + sorted_idx_t2i_base[q_idx, TOP_K_T2I:].tolist()
    sorted_idx_t2i_final[q_idx, :TOP_K_T2I] = torch.tensor(new_top_5, device=device)

cache_t2i.save()

# --- Calcul du temps projeté ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# --- Bilan T2I ---
evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_ensemble.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_ensemble.md'),
    temps_vlm=temps_total_vlm_t2i
)

# --- Autopsie T2I ---
_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=[True]*limit_t2i)
'''

'\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE T2I (POINTWISE SOTA + PROMPT ENSEMBLING)")\nprint("="*50)\n\n# 1. Désactivation du LoRA\nif hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):\n    print("⚙️ Désactivation des adaptateurs LoRA pour test Pointwise Zero-Shot...")\n    vlm.model.disable_adapters()\nelse:\n    print("✅ Le modèle est bien en mode de base (Zero-Shot).")\n\n# 2. Initialisation d\'un nouveau cache pour l\'Ensembling\nCACHE_ENSEMBLE_FILE = config.CACHE_T2I_FILE.replace(\'.json\', \'_ensemble.json\')\ncache_t2i = RerankingCache(CACHE_ENSEMBLE_FILE)\n\nsorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)\nsorted_idx_t2i_final = sorted_idx_t2i_base.clone()\n\ntemps_vlm_reel = 0.0\nappels_vlm_reels = 0\nALPHA = 0.5 \n\nfor q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Prompt Ensembling"):\n    top_5 = sorted_idx_t2i_base[q_idx, :TOP_K_T2I].tolist()\n\n    # Vérification du cache\n    cached = cache_t2

### Pointwise Top 5 (Sans cascade, Late Fusion α)

In [8]:
'''
print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (FULL POINTWISE SOTA - Sans Cascade)")
print("="*50)

# 1. Désactivation du LoRA (Le Pointwise fonctionne mieux sur le modèle de base non biaisé)
if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA pour test Pointwise Zero-Shot...")
    vlm.model.disable_adapters()
else:
    print("✅ Le modèle est bien en mode de base (Zero-Shot).")

# 2. Initialisation
cache_t2i = RerankingCache(config.CACHE_T2I_FILE)
sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0

# ⚖️ Poids de la fusion (0.5 = 50% MIRAGE / 50% VLM)
ALPHA = 0.5 

# Comme on envoie TOUT, le mask est 100% True
mask_vlm_called = [True] * limit_t2i

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Full Pointwise"):
    top_5 = sorted_idx_t2i_base[q_idx, :TOP_K_T2I].tolist()
    
    # Vérification du cache
    cached = cache_t2i.get(q_idx)
    if cached:
        new_top_5 = cached
    else:
        images_pil = [dataset[idx]['image'] for idx in top_5]
        requete = all_texts[q_idx]
        
        # ⚡ VERSION BATCHÉE T2I ⚡
        prompt = f"Does the following image exactly and perfectly match this description: '{requete}'? Look at every tiny detail. Answer strictly with 'Yes' or 'No'."
        prompts_batch = [prompt] * 5
        
        t0_call = time.time()
        
        # On score les 5 images D'UN COUP avec le même texte
        scores_vlm = vlm.score_image_pointwise_batch(prompts_batch, images_pil)
        
        t1_call = time.time()

        temps_vlm_reel += (t1_call - t0_call)
        appels_vlm_reels += 1 # On compte ça comme 1 requête complète traitée

        # --- LATE FUSION ---
        # 1. On récupère les scores bruts de MIRAGE pour le top 5
        scores_mirage_bruts = [S_t2i_stage1[q_idx, idx].item() for idx in top_5]
        
        # 2. Normalisation Min-Max locale des scores MIRAGE pour qu'ils soient entre 0 et 1
        max_s = max(scores_mirage_bruts)
        min_s = min(scores_mirage_bruts)
        if max_s > min_s:
            scores_mirage_norm = [(s - min_s) / (max_s - min_s) for s in scores_mirage_bruts]
        else:
            scores_mirage_norm = [1.0] * 5

        # 3. Fusion mathématique (Alpha * VLM + (1-Alpha) * MIRAGE)
        final_scores = []
        for i, idx in enumerate(top_5):
            score_mixte = ((1 - ALPHA) * scores_mirage_norm[i]) + (ALPHA * scores_vlm[i])
            final_scores.append((score_mixte, idx))
            
        # 4. Nouveau tri basé sur le score fusionné
        final_scores.sort(key=lambda x: x[0], reverse=True)
        new_top_5 = [idx for score, idx in final_scores]
        
        # Mise en cache
        cache_t2i.set(q_idx, new_top_5)
        if appels_vlm_reels % 5 == 0: 
            cache_t2i.save()
            
    ordre_final = new_top_5 + top_5[TOP_K_T2I:] if len(top_5) > TOP_K_T2I else new_top_5 + sorted_idx_t2i_base[q_idx, TOP_K_T2I:].tolist()
    sorted_idx_t2i_final[q_idx, :TOP_K_T2I] = torch.tensor(new_top_5, device=device)

cache_t2i.save()

# --- Calcul du temps projeté ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# --- Bilan T2I (Métriques et Sauvegarde) ---
evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv, 
    md_path=config.metriques_t2i_md,
    temps_vlm=temps_total_vlm_t2i
)

# --- Autopsie T2I (Sauvetages/Sabotages) ---
_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=mask_vlm_called)
'''

'\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE T2I (FULL POINTWISE SOTA - Sans Cascade)")\nprint("="*50)\n\n# 1. Désactivation du LoRA (Le Pointwise fonctionne mieux sur le modèle de base non biaisé)\nif hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):\n    print("⚙️ Désactivation des adaptateurs LoRA pour test Pointwise Zero-Shot...")\n    vlm.model.disable_adapters()\nelse:\n    print("✅ Le modèle est bien en mode de base (Zero-Shot).")\n\n# 2. Initialisation\ncache_t2i = RerankingCache(config.CACHE_T2I_FILE)\nsorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)\nsorted_idx_t2i_final = sorted_idx_t2i_base.clone()\n\ntemps_vlm_reel = 0.0\nappels_vlm_reels = 0\n\n# ⚖️ Poids de la fusion (0.5 = 50% MIRAGE / 50% VLM)\nALPHA = 0.5 \n\n# Comme on envoie TOUT, le mask est 100% True\nmask_vlm_called = [True] * limit_t2i\n\nfor q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Full Pointwise"):\n    top_5 = sorted_idx_t2i_base

### Listwise LoRA (Juge Final sur Top 5)

In [9]:
# Tressssss long
'''
print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I ÉTAPE 3 : JUGE FINAL LISTWISE (LoRA)")
print("="*50)

# 1. On utilise le tenseur final de l'étape précédente (Top 10) comme nouveau point de départ
sorted_idx_t2i_stage2 = sorted_idx_t2i_final.clone()
sorted_idx_t2i_stage3 = sorted_idx_t2i_stage2.clone()

# 2. ⚙️ CHARGEMENT DU MODÈLE ELITE (LoRA)

if os.path.exists(config.LORA_OUTPUT_DIR):
    print(f"⚙️ Injection des poids LoRA ELITE CoT pour l'évaluation Listwise...")
    if not (getattr(vlm.model, "_hf_peft_config_loaded", False) or isinstance(vlm.model, PeftModel)):
        old_no_split = getattr(vlm.model, "_no_split_modules", None)
        vlm.model._no_split_modules = None
        vlm.model = PeftModel.from_pretrained(vlm.model, config.LORA_OUTPUT_DIR)
        vlm.model._no_split_modules = old_no_split
    vlm.model.eval()
    print("✅ Modèle Listwise prêt !")
else:
    print("⚠️ Attention : Dossier LoRA introuvable, le modèle tournera en Zero-Shot.")

# 3. Initialisation du cache pour cette 3ème étape
CACHE_STAGE3_FILE = config.CACHE_T2I_FILE.replace('.json', '_stage3_listwise.json')
cache_stage3 = RerankingCache(CACHE_STAGE3_FILE)

temps_vlm_reel = 0.0
appels_vlm_reels = 0

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Stage 3 (Juge Final)"):
    # On ne prend QUE le Top 5 de la phase précédente (qui a déjà été pré-trié par le Pointwise)
    top_5 = sorted_idx_t2i_stage2[q_idx, :5].tolist()
    les_suivants = sorted_idx_t2i_stage2[q_idx, 5:].tolist()
    
    cached = cache_stage3.get(q_idx)
    if cached:
        new_top_5 = cached
    else:
        requete_texte = all_texts[q_idx]
        images_pil = [dataset[idx]['image'] for idx in top_5]
        
        # Mapping des positions (1 à 5) vers les vrais IDs d'images
        pos_to_id = {i+1: top_5[i] for i in range(5)}
        
        # 📝 PROMPT LISTWISE (Adapté pour comparer 5 images)
        prompt = (
            f"You are an expert visual verifier. These 5 images are pre-ranked by an AI for the query: '{requete_texte}'. "
            f"Image 1 is mathematically the most probable match. Your task is to verify this. "
            f"Compare the images directly. Do NOT demote Image 1 unless another image clearly matches the subtle details better. "
            f"Think step by step, penalize visual hallucinations, and output the Final Ranking."
            f"\nFormat:\nFinal Ranking: [PositionA, PositionB, PositionC, PositionD, PositionE]"
        )
        
        t0_call = time.time()
        response_text = vlm.generate_response(prompt, images_pil) 
        t1_call = time.time()
        
        temps_vlm_reel += (t1_call - t0_call)
        appels_vlm_reels += 1
        
        # --- PARSING ---
        match = re.search(r'Final Ranking:\s*(?:\[)?(.*?)(?:\])?$', response_text, re.IGNORECASE | re.MULTILINE)
        predicted_positions = None
        if match:
            nums = [int(n) for n in re.findall(r'\d+', match.group(1))]
            predicted_positions = []
            for x in nums:
                if 1 <= x <= 5 and x not in predicted_positions:
                    predicted_positions.append(x)
            missing = [i for i in range(1, 6) if i not in predicted_positions]
            predicted_positions.extend(missing)
        
        # Fallback de sécurité : Si le parsing échoue, on garde l'ordre de l'étape 2
        if predicted_positions is None or len(predicted_positions) != 5:
            new_top_5 = top_5
        else:
            # Reconversion des positions (1, 2, 3...) en IDs réels
            new_top_5 = [pos_to_id[pos] for pos in predicted_positions]
            
        # Mise en cache
        cache_stage3.set(q_idx, new_top_5)
        if appels_vlm_reels % 5 == 0:
            cache_stage3.save()
            
    # Mise à jour du tenseur avec le nouvel ordre
    ordre_final = new_top_5 + les_suivants
    sorted_idx_t2i_stage3[q_idx, :len(ordre_final)] = torch.tensor(ordre_final, device=device)

cache_stage3.save()

temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# =====================================================================
# ÉVALUATION GLOBALE (Stage 1 vs Stage 3)
# =====================================================================
print("\n" + "="*80)
print("🏆 COMPARAISON FINALE : MIRAGE (Base) vs PIPELINE 3 ÉTAGES (Coarse-to-Fine)")
print("="*80)

# On compare l'état initial (MIRAGE pur) avec l'état final (Après Pointwise + Listwise)
evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_stage3[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_stage3_final.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_stage3_final.md'),
    temps_vlm=temps_total_vlm_t2i
)

# Autopsie globale : on veut voir combien d'erreurs de MIRAGE on a corrigées au total
_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_stage3, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=[True]*limit_t2i)
'''

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1709685/1862032654.py:2: SyntaxWarning: invalid escape sequence '\s'
  '''


'\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE T2I ÉTAPE 3 : JUGE FINAL LISTWISE (LoRA)")\nprint("="*50)\n\n# 1. On utilise le tenseur final de l\'étape précédente (Top 10) comme nouveau point de départ\nsorted_idx_t2i_stage2 = sorted_idx_t2i_final.clone()\nsorted_idx_t2i_stage3 = sorted_idx_t2i_stage2.clone()\n\n# 2. ⚙️ CHARGEMENT DU MODÈLE ELITE (LoRA)\n\nif os.path.exists(config.LORA_OUTPUT_DIR):\n    print(f"⚙️ Injection des poids LoRA ELITE CoT pour l\'évaluation Listwise...")\n    if not (getattr(vlm.model, "_hf_peft_config_loaded", False) or isinstance(vlm.model, PeftModel)):\n        old_no_split = getattr(vlm.model, "_no_split_modules", None)\n        vlm.model._no_split_modules = None\n        vlm.model = PeftModel.from_pretrained(vlm.model, config.LORA_OUTPUT_DIR)\n        vlm.model._no_split_modules = old_no_split\n    vlm.model.eval()\n    print("✅ Modèle Listwise prêt !")\nelse:\n    print("⚠️ Attention : Dossier LoRA introuvable, le modèle tournera en Zero-Shot.")\n\n# 3. I

### Pointwise CoT (Avec Cascade & Late Fusion α)

In [10]:
'''
# ==========================================
# FONCTION DE PARSING POUR LE CoT (Score sur 100)
# ==========================================
def parse_cot_score(response_text):
    """Extrait la note finale (Score: X) générée par le VLM."""
    match = re.search(r"Score\s*:\s*(\d+)", response_text, re.IGNORECASE)
    if match:
        return float(match.group(1)) / 100.0 
    return 0.0 

print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (POINTWISE CoT + CASCADE LATE FUSION)")
print("="*50)

TOP_K_TEST = 10

if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA...")
    vlm.model.disable_adapters()

# 1. Calibrage du seuil de confiance
print("🎯 Calibrage du seuil de confiance...")
seuil_t2i = calibrate_confidence_threshold(S_t2i_val_stage1, targets_t2i_val_gpu, is_i2t=False, target_recall=0.85)

CACHE_TOP10_COT_FILE = config.CACHE_T2I_FILE.replace('.json', '_top10_cot_cascade.json')
cache_t2i_cot = RerankingCache(CACHE_TOP10_COT_FILE)

sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0
mask_vlm_called = [False] * limit_t2i
requetes_sauvees = 0

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Pointwise CoT + Cascade"):
    top_10 = sorted_idx_t2i_base[q_idx, :TOP_K_TEST].tolist()
    
    # --- CASCADE RERANKING ---
    score_top1 = S_t2i_stage1[q_idx, top_10[0]].item()
    score_top2 = S_t2i_stage1[q_idx, top_10[1]].item()
    ecart_confiance = score_top1 - score_top2
    
    if ecart_confiance > seuil_t2i:
        new_top_10 = top_10
        requetes_sauvees += 1
    else:
        mask_vlm_called[q_idx] = True
        cached = cache_t2i_cot.get(q_idx)
        if cached:
            new_top_10 = cached
        else:
            requete = all_texts[q_idx]
            
            # PROMPT AVEC CHAIN OF THOUGHT
            prompt = f"""Analyze this image step-by-step against the description: '{requete}'. 
            Check every object, action, and attribute mentioned. List any missing or conflicting details. 
            Finally, rate the match on a scale from 0 to 100 on a new line exactly like this: 'Score: X'."""
            
            scores_vlm = []
            
            # Évaluation POINTWISE (1 image à la fois = très peu de VRAM)
            for idx in top_10:
                img = dataset[idx]['image']
                
                t0_call = time.time()
                response_text = vlm.generate_response(prompt, [img])
                t1_call = time.time()
                
                temps_vlm_reel += (t1_call - t0_call)
                appels_vlm_reels += 1
                
                score = parse_cot_score(response_text)
                scores_vlm.append(score)
            
            # --- LATE FUSION DOUCE ---
            # Au lieu d'éliminer cash (tournoi), on combine les certitudes
            scores_mirage_bruts = [S_t2i_stage1[q_idx, idx].item() for idx in top_10]
            max_s = max(scores_mirage_bruts)
            min_s = min(scores_mirage_bruts)
            scores_mirage_norm = [(s - min_s) / (max_s - min_s) if max_s > min_s else 1.0 for s in scores_mirage_bruts]

            # Alpha fixe (ou dynamique) : on donne 50% de poids à MIRAGE et 50% au VLM
            ALPHA = 0.5 
            final_scores = []
            for i, idx in enumerate(top_10):
                score_mixte = ((1 - ALPHA) * scores_mirage_norm[i]) + (ALPHA * scores_vlm[i])
                final_scores.append((score_mixte, idx))
                
            final_scores.sort(key=lambda x: x[0], reverse=True)
            new_top_10 = [idx for score, idx in final_scores]
            
            cache_t2i_cot.set(q_idx, new_top_10)
            if appels_vlm_reels % 30 == 0:
                cache_t2i_cot.save()
            
    sorted_idx_t2i_final[q_idx, :TOP_K_TEST] = torch.tensor(new_top_10, device=device)

cache_t2i_cot.save()

print(f"\n✅ Économie Cascade : {requetes_sauvees}/{limit_t2i} requêtes sans appel VLM !")

# --- Bilan ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_cot_cascade.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_cot_cascade.md'),
    temps_vlm=temps_total_vlm_t2i
)

_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=mask_vlm_called)
'''

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1709685/1379757331.py:1: SyntaxWarning: invalid escape sequence '\s'
  '''


'\n# ==========================================\n# FONCTION DE PARSING POUR LE CoT (Score sur 100)\n# ==========================================\ndef parse_cot_score(response_text):\n    """Extrait la note finale (Score: X) générée par le VLM."""\n    match = re.search(r"Score\\s*:\\s*(\\d+)", response_text, re.IGNORECASE)\n    if match:\n        return float(match.group(1)) / 100.0 \n    return 0.0 \n\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE T2I (POINTWISE CoT + CASCADE LATE FUSION)")\nprint("="*50)\n\nTOP_K_TEST = 10\n\nif hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):\n    print("⚙️ Désactivation des adaptateurs LoRA...")\n    vlm.model.disable_adapters()\n\n# 1. Calibrage du seuil de confiance\nprint("🎯 Calibrage du seuil de confiance...")\nseuil_t2i = calibrate_confidence_threshold(S_t2i_val_stage1, targets_t2i_val_gpu, is_i2t=False, target_recall=0.85)\n\nCACHE_TOP10_COT_FILE = config.CACHE_T2I_FILE.replace(\'.json\', \'_top10_cot

### Tournoi Pairwise (Avec Cascade)


In [11]:
'''
# ==========================================
# FONCTION DE PARSING POUR LE TOURNOI
# ==========================================
def parse_pairwise_winner(response_text):
    """Extrait le choix du VLM (Image A ou Image B)."""
    # On cherche une réponse claire 'A' ou 'B'
    match = re.search(r'\b(A|B)\b', response_text.upper())
    if match:
        return match.group(1)
    # En cas d'hallucination ou de refus, on garde le champion en titre (A par défaut)
    return 'A'

print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (PAIRWISE TOURNAMENT + CASCADE CONFIDENCE - TOP 10)")
print("="*50)

TOP_K_TEST = 10

if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA pour test Pairwise Zero-Shot...")
    vlm.model.disable_adapters()

# 1. Calibrage du seuil sur le set de validation (On cible 85% des erreurs)
print("🎯 Calibrage du seuil de confiance...")
seuil_t2i = calibrate_confidence_threshold(S_t2i_val_stage1, targets_t2i_val_gpu, is_i2t=False, target_recall=0.85)

# Nouveau cache pour l'approche Tournoi
CACHE_TOP10_PAIR_FILE = config.CACHE_T2I_FILE.replace('.json', '_top10_pairwise.json')
cache_t2i_pair = RerankingCache(CACHE_TOP10_PAIR_FILE)

sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0

# Par défaut, on ne fait pas appel au VLM (on mettra True que si on a un doute)
mask_vlm_called = [False] * limit_t2i
requetes_sauvees = 0

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Pairwise (Tournoi)"):
    top_10 = sorted_idx_t2i_base[q_idx, :TOP_K_TEST].tolist()
    
    # --- CASCADE RERANKING : Vérification de la confiance ---
    score_top1 = S_t2i_stage1[q_idx, top_10[0]].item()
    score_top2 = S_t2i_stage1[q_idx, top_10[1]].item()
    ecart_confiance = score_top1 - score_top2
    
    # Si l'écart est supérieur au seuil, on fait aveuglément confiance à la Phase 1
    if ecart_confiance > seuil_t2i:
        new_top_10 = top_10
        requetes_sauvees += 1
    else:
        # La Phase 1 hésite, on lance le VLM
        mask_vlm_called[q_idx] = True
        
        # Vérification du cache
        cached = cache_t2i_pair.get(q_idx)
        if cached:
            new_top_10 = cached
        else:
            requete = all_texts[q_idx]
            
            # Le champion initial est le Top 1 de MIRAGE
            champion_idx = top_10[0]
            champion_img = dataset[champion_idx]['image']
            
            # Le tournoi commence (du Top 2 au Top 10)
            for i in range(1, TOP_K_TEST):
                challenger_idx = top_10[i]
                challenger_img = dataset[challenger_idx]['image']
                
                prompt = f"""You are an expert evaluator. I am looking for the image that BEST matches this description: '{requete}'
                
                Image A is the first image. Image B is the second image.
                Compare both images carefully against every detail of the text. 
                Which image is a better match? Answer STRICTLY with a single letter: 'A' or 'B'."""
                
                t0_call = time.time()
                
                # On envoie les 2 images au VLM (A = champion, B = challenger)
                response_text = vlm.generate_response(prompt, [champion_img, challenger_img])
                
                t1_call = time.time()
                temps_vlm_reel += (t1_call - t0_call)
                appels_vlm_reels += 1
                
                winner = parse_pairwise_winner(response_text)
                
                # Si le challenger (B) gagne, il devient le nouveau champion
                if winner == 'B':
                    champion_idx = challenger_idx
                    champion_img = challenger_img
            
            # --- RECONSTRUCTION DU CLASSEMENT ---
            # Le champion prend la 1ère place. 
            # Les autres gardent leur ordre relatif dicté par MIRAGE (Phase 1).
            new_top_10 = [champion_idx]
            for idx in top_10:
                if idx != champion_idx:
                    new_top_10.append(idx)
                    
            # Mise en cache
            cache_t2i_pair.set(q_idx, new_top_10)
            if appels_vlm_reels % 45 == 0: # Sauvegarde régulière (toutes les ~5 requêtes complètes)
                cache_t2i_pair.save()
            
    # Mise à jour du tenseur final
    sorted_idx_t2i_final[q_idx, :TOP_K_TEST] = torch.tensor(new_top_10, device=device)

cache_t2i_pair.save()

print(f"\n✅ Économie Cascade : {requetes_sauvees}/{limit_t2i} requêtes ont été traitées sans appeler le VLM !")

# --- Bilan T2I ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_top10_pairwise_cascade.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_top10_pairwise_cascade.md'),
    temps_vlm=temps_total_vlm_t2i
)

_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=mask_vlm_called)
'''

'\n# ==========================================\n# FONCTION DE PARSING POUR LE TOURNOI\n# ==========================================\ndef parse_pairwise_winner(response_text):\n    """Extrait le choix du VLM (Image A ou Image B)."""\n    # On cherche une réponse claire \'A\' ou \'B\'\n    match = re.search(r\'\x08(A|B)\x08\', response_text.upper())\n    if match:\n        return match.group(1)\n    # En cas d\'hallucination ou de refus, on garde le champion en titre (A par défaut)\n    return \'A\'\n\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE T2I (PAIRWISE TOURNAMENT + CASCADE CONFIDENCE - TOP 10)")\nprint("="*50)\n\nTOP_K_TEST = 10\n\nif hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):\n    print("⚙️ Désactivation des adaptateurs LoRA pour test Pairwise Zero-Shot...")\n    vlm.model.disable_adapters()\n\n# 1. Calibrage du seuil sur le set de validation (On cible 85% des erreurs)\nprint("🎯 Calibrage du seuil de confiance...")\nseuil_t2i = c

### Pointwise Top 10 (Sans cascade, Late Fusion α)


In [12]:
'''
# %% [markdown]
# ## Reranking T2I (FULL POINTWISE SOTA - Top 10)

# %%
print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (FULL POINTWISE SOTA - TOP 10)")
print("="*50)

# 2. Désactivation du LoRA
if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA pour test Pointwise Zero-Shot...")
    vlm.model.disable_adapters()
else:
    print("✅ Le modèle est bien en mode de base (Zero-Shot).")

# 3. Initialisation avec un nouveau cache spécifique au Top 10
CACHE_TOP10_FILE = config.CACHE_T2I_FILE.replace('.json', '_top10.json')
cache_t2i = RerankingCache(CACHE_TOP10_FILE)

sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0
ALPHA = 0.5 

mask_vlm_called = [True] * limit_t2i

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Full Pointwise (Top 10)"):
    top_10 = sorted_idx_t2i_base[q_idx, :TOP_K_T2I].tolist()
    
    # Vérification du cache
    cached = cache_t2i.get(q_idx)
    if cached:
        new_top_10 = cached
    else:
        images_pil = [dataset[idx]['image'] for idx in top_10]
        requete = all_texts[q_idx]
        
        prompt = f"Does the following image exactly and perfectly match this description: '{requete}'? Look at every tiny detail. Answer strictly with 'Yes' or 'No'."
        
        t0_call = time.time()
        
        # 🛡️ SÉCURITÉ VRAM : On coupe le batch de 10 en deux batchs de 5
        scores_vlm_part1 = vlm.score_image_pointwise_batch([prompt] * 5, images_pil[:5])
        scores_vlm_part2 = vlm.score_image_pointwise_batch([prompt] * 5, images_pil[5:])
        scores_vlm = scores_vlm_part1 + scores_vlm_part2 # On recolle les 10 scores
        
        t1_call = time.time()

        temps_vlm_reel += (t1_call - t0_call)
        appels_vlm_reels += 1 

        # --- LATE FUSION ---
        # 1. On récupère les 10 scores bruts de MIRAGE
        scores_mirage_bruts = [S_t2i_stage1[q_idx, idx].item() for idx in top_10]
        
        # 2. Normalisation Min-Max
        max_s = max(scores_mirage_bruts)
        min_s = min(scores_mirage_bruts)
        if max_s > min_s:
            scores_mirage_norm = [(s - min_s) / (max_s - min_s) for s in scores_mirage_bruts]
        else:
            scores_mirage_norm = [1.0] * 10

        # 3. Fusion mathématique (Alpha * VLM + (1-Alpha) * MIRAGE)
        final_scores = []
        for i, idx in enumerate(top_10):
            score_mixte = ((1 - ALPHA) * scores_mirage_norm[i]) + (ALPHA * scores_vlm[i])
            final_scores.append((score_mixte, idx))
            
        # 4. Nouveau tri basé sur le score fusionné
        final_scores.sort(key=lambda x: x[0], reverse=True)
        new_top_10 = [idx for score, idx in final_scores]
        
        # Mise en cache
        cache_t2i.set(q_idx, new_top_10)
        if appels_vlm_reels % 5 == 0: 
            cache_t2i.save()
            
    # Mise à jour du tenseur final
    ordre_final = new_top_10 + top_10[TOP_K_T2I:] if len(top_10) > TOP_K_T2I else new_top_10 + sorted_idx_t2i_base[q_idx, TOP_K_T2I:].tolist()
    sorted_idx_t2i_final[q_idx, :TOP_K_T2I] = torch.tensor(new_top_10, device=device)

cache_t2i.save()

# --- Calcul du temps projeté ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# --- Bilan T2I ---
evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_top10.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_top10.md'),
    temps_vlm=temps_total_vlm_t2i
)

# --- Autopsie T2I ---
_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=mask_vlm_called)
'''

'\n# %% [markdown]\n# ## Reranking T2I (FULL POINTWISE SOTA - Top 10)\n\n# %%\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE T2I (FULL POINTWISE SOTA - TOP 10)")\nprint("="*50)\n\n# 2. Désactivation du LoRA\nif hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):\n    print("⚙️ Désactivation des adaptateurs LoRA pour test Pointwise Zero-Shot...")\n    vlm.model.disable_adapters()\nelse:\n    print("✅ Le modèle est bien en mode de base (Zero-Shot).")\n\n# 3. Initialisation avec un nouveau cache spécifique au Top 10\nCACHE_TOP10_FILE = config.CACHE_T2I_FILE.replace(\'.json\', \'_top10.json\')\ncache_t2i = RerankingCache(CACHE_TOP10_FILE)\n\nsorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)\nsorted_idx_t2i_final = sorted_idx_t2i_base.clone()\n\ntemps_vlm_reel = 0.0\nappels_vlm_reels = 0\nALPHA = 0.5 \n\nmask_vlm_called = [True] * limit_t2i\n\nfor q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Full Pointwise (Top 10)"):\n  

In [13]:
'''
# Fichier de stockage des scores bruts
RAW_SCORES_FILE = os.path.join(config.RESULTS_DIR, "scores_bruts_t2i.json")

# On récupère le Top 10 de base
sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
scores_all_queries = {}

# Chargement du cache existant pour ne pas recalculer si vous avez déjà des résultats
if os.path.exists(RAW_SCORES_FILE):
    with open(RAW_SCORES_FILE, 'r') as f:
        scores_all_queries = json.load(f)

for q_idx in tqdm(range(limit_t2i), desc="Extraction des scores bruts"):
    str_qidx = str(q_idx)
    top_10 = sorted_idx_t2i_base[q_idx, :10].tolist()
    
    if str_qidx in scores_all_queries:
        continue

    # 1. Récupération des images et de la requête
    images_pil = [dataset[idx]['image'] for idx in top_10]
    requete = all_texts[q_idx]
    prompt = f"Does the following image exactly and perfectly match this description: '{requete}'? Answer strictly with 'Yes' or 'No'."
    
    # 2. Inférence VLM (Scoring Pointwise par Batch)
    # On coupe en deux batchs de 5 pour la VRAM
    scores_vlm_p1 = vlm.score_image_pointwise_batch([prompt] * 5, images_pil[:5])
    scores_vlm_p2 = vlm.score_image_pointwise_batch([prompt] * 5, images_pil[5:])
    scores_vlm = scores_vlm_p1 + scores_vlm_p2
    
    # 3. Récupération des scores MIRAGE bruts
    scores_mirage = [S_t2i_stage1[q_idx, idx].item() for idx in top_10]
    
    # Sauvegarde des données brutes
    scores_all_queries[str_qidx] = {
        "candidate_ids": top_10,
        "vlm_scores": scores_vlm,     # Probabilités [0, 1]
        "mirage_scores": scores_mirage # Scores de similarité phase 1
    }
    
    if q_idx % 10 == 0:
        with open(RAW_SCORES_FILE, 'w') as f:
            json.dump(scores_all_queries, f)

with open(RAW_SCORES_FILE, 'w') as f:
    json.dump(scores_all_queries, f)

print(f"✅ Scores bruts sauvegardés dans : {RAW_SCORES_FILE}")

'''

'\n# Fichier de stockage des scores bruts\nRAW_SCORES_FILE = os.path.join(config.RESULTS_DIR, "scores_bruts_t2i.json")\n\n# On récupère le Top 10 de base\nsorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)\nscores_all_queries = {}\n\n# Chargement du cache existant pour ne pas recalculer si vous avez déjà des résultats\nif os.path.exists(RAW_SCORES_FILE):\n    with open(RAW_SCORES_FILE, \'r\') as f:\n        scores_all_queries = json.load(f)\n\nfor q_idx in tqdm(range(limit_t2i), desc="Extraction des scores bruts"):\n    str_qidx = str(q_idx)\n    top_10 = sorted_idx_t2i_base[q_idx, :10].tolist()\n\n    if str_qidx in scores_all_queries:\n        continue\n\n    # 1. Récupération des images et de la requête\n    images_pil = [dataset[idx][\'image\'] for idx in top_10]\n    requete = all_texts[q_idx]\n    prompt = f"Does the following image exactly and perfectly match this description: \'{requete}\'? Answer strictly with \'Yes\' or \'No\'."\n\n    # 2. Inférence VL

In [14]:
'''

def evaluate_fusion(scores_dict, targets_gpu, method="additive", alpha=0.5, rrf_k=60, top_k=10, cascade_threshold=None):
    """Calcule le R@1 en simulant dynamiquement le Top-K et la Cascade."""
    correct_count = 0
    total = len(scores_dict)
    reranked_results = {}
    mask_vlm_called = {} # Pour garder la trace des requêtes sauvées par la cascade
    
    for q_idx_str, data in scores_dict.items():
        q_idx = int(q_idx_str)
        target = targets_gpu[q_idx].item()
        
        ids = data["candidate_ids"][:top_k]
        vlm = np.array(data["vlm_scores"][:top_k])
        mirage = np.array(data["mirage_scores"][:top_k])
        
        # 🛡️ --- LOGIQUE DE CASCADE --- 🛡️
        ecart_confiance = mirage[0] - mirage[1] if len(mirage) > 1 else 0
        
        if cascade_threshold is not None and ecart_confiance > cascade_threshold:
            # MIRAGE est sûr de lui : on valide sans utiliser le VLM
            new_top_k = ids
            mask_vlm_called[q_idx] = False
        else:
            # MIRAGE hésite : on fait la fusion
            mask_vlm_called[q_idx] = True
            mirage_norm = (mirage - mirage.min()) / (mirage.max() - mirage.min() + 1e-8)
            
            if method == "moyenne":
                final_scores = (alpha * vlm) + ((1 - alpha) * mirage_norm)
            elif method == "multiplicative":
                final_scores = vlm * mirage_norm
            elif method == "maximum":
                final_scores = np.maximum(vlm, mirage_norm)
            elif method == "concatenation":
                final_scores = (vlm * 1000.0) + mirage_norm
            elif method == "rrf":
                rank_vlm = np.argsort(np.argsort(-vlm)) + 1
                rank_mirage = np.argsort(np.argsort(-mirage_norm)) + 1
                final_scores = (1.0 / (rrf_k + rank_vlm)) + (1.0 / (rrf_k + rank_mirage))
                
            best_indices = np.argsort(-final_scores) 
            new_top_k = [ids[i] for i in best_indices]
            
        reranked_results[q_idx] = new_top_k
        if new_top_k[0] == target:
            correct_count += 1
            
    return correct_count / total, reranked_results, mask_vlm_called

# ==========================================
# 1. PRÉPARATION DES DONNÉES ET DES TESTS
# ==========================================
RAW_SCORES_FILE = os.path.join(config.RESULTS_DIR, "scores_bruts_t2i.json")

with open(RAW_SCORES_FILE, 'r') as f:
    scores_bruts = json.load(f)

# 🛠️ AJOUT ICI : On recalcule le classement de base de la Phase 1 et la limite
sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)

print("\n" + "="*70)
print("🧪 LABORATOIRE DE FUSION : GÉNÉRATION DU TABLEAU COMPARATIF")
print("="*70)

options_top_k = [5, 10]
methodes_simples = ["multiplicative", "maximum", "concatenation", "rrf"]

# 🎯 NOUVEAUTÉ : Grid Search sur le Seuil de Cascade
print("🎯 Calibrage des multiples seuils de cascade sur le set de validation...")
options_cascade = {"Sans Cascade": None}

# On teste plusieurs niveaux d'exigence (de très permissif à très strict)
recalls_a_tester = [0.70, 0.80, 0.85, 0.90, 0.95, 0.99, 1.00]

for tr in recalls_a_tester:
    # Utilise S_t2i_val_stage1 et targets_t2i_val_gpu (déjà chargés dans ton notebook)
    seuil = calibrate_confidence_threshold(S_t2i_val_stage1, targets_t2i_val_gpu, is_i2t=False, target_recall=tr)
    
    # On ajoute ce seuil au dictionnaire des cascades à tester
    nom_cascade = f"Avec (Recall={tr:.2f})"
    options_cascade[nom_cascade] = seuil

resultats_tableau = []
grand_gagnant = {"r1": 0, "nom": "", "dict": {}, "mask": {}}

# ==========================================
# 2. EXÉCUTION DE TOUTES LES COMBINAISONS
# ==========================================
# On désactive l'affichage de tqdm pour que le tableau s'affiche proprement à la fin
for top_k in options_top_k:
    for nom_cascade, val_cascade in options_cascade.items():
        
        # 1. Tests des méthodes simples
        for methode in methodes_simples:
            r1, res_dict, mask = evaluate_fusion(scores_bruts, targets_t2i_gpu, method=methode, top_k=top_k, cascade_threshold=val_cascade)
            req_sauvees = sum(not v for v in mask.values())
            
            resultats_tableau.append({
                "Profondeur": f"Top {top_k}",
                "Cascade": nom_cascade.split(" ")[0], # Juste "Sans" ou "Avec"
                "Stratégie de Fusion": methode.capitalize(),
                "R@1 (%)": r1 * 100,
                "VLM Évités": req_sauvees
            })
            
            if r1 > grand_gagnant["r1"]:
                grand_gagnant = {"r1": r1, "nom": f"{methode.capitalize()}_Top{top_k}_{nom_cascade.split(' ')[0]}", "dict": res_dict, "mask": mask}

        # 2. Test Additif avec Grid Search Intégré
        best_add_r1, best_alpha, best_add_dict, best_add_mask = 0, 0, {}, {}
        for a in np.linspace(0, 1, 21):
            r1, res_dict, mask = evaluate_fusion(scores_bruts, targets_t2i_gpu, method="moyenne", alpha=a, top_k=top_k, cascade_threshold=val_cascade)
            if r1 > best_add_r1:
                best_add_r1 = r1
                best_alpha = a
                best_add_dict = res_dict
                best_add_mask = mask
                
        req_sauvees_add = sum(not v for v in best_add_mask.values())
        resultats_tableau.append({
            "Profondeur": f"Top {top_k}",
            "Cascade": nom_cascade.split(" ")[0],
            "Stratégie de Fusion": f"Grid Search (Alpha={best_alpha:.2f})",
            "R@1 (%)": best_add_r1 * 100,
            "VLM Évités": req_sauvees_add
        })
        
        if best_add_r1 > grand_gagnant["r1"]:
            grand_gagnant = {"r1": best_add_r1, "nom": f"Grid_Search_A{best_alpha:.2f}_Top{top_k}_{nom_cascade.split(' ')[0]}", "dict": best_add_dict, "mask": best_add_mask}

# ==========================================
# 3. AFFICHAGE DU CLASSEMENT
# ==========================================
# Création du DataFrame Pandas et tri par les meilleurs scores R@1
df_resultats = pd.DataFrame(resultats_tableau)
df_resultats = df_resultats.sort_values(by=["R@1 (%)", "VLM Évités"], ascending=[False, False]).reset_index(drop=True)

print("\n🏆 CLASSEMENT DES STRATÉGIES DE RERANKING :\n")
display(df_resultats) # Affiche un beau tableau HTML dans Jupyter

print("\n" + "⭐ "*15)
print(f"LE GRAND GAGNANT ABSOLU EST : {grand_gagnant['nom']}")
print(f"R@1 : {grand_gagnant['r1']*100:.2f} %")
print("⭐ "*15 + "\n")

# ==========================================
# 4. RECONSTRUCTION ET AUTOPSIE DU GAGNANT
# ==========================================
print(f"⚙️ Sauvegarde et Autopsie de la meilleure configuration ({grand_gagnant['nom']})...")

sorted_idx_t2i_final = sorted_idx_t2i_base.clone()
for q_idx_str, new_top_k in grand_gagnant["dict"].items():
    q_idx = int(q_idx_str)
    ordre_final = new_top_k + sorted_idx_t2i_base[q_idx, len(new_top_k):].tolist()
    sorted_idx_t2i_final[q_idx, :len(ordre_final)] = torch.tensor(ordre_final, device=device)

# Reconstruction de la liste mask_vlm_called pour l'autopsie
mask_vlm_called_list = [False] * limit_t2i
for q_idx_str, was_called in grand_gagnant["mask"].items():
    mask_vlm_called_list[int(q_idx_str)] = was_called

# Nommage propre du fichier CSV
file_suffix = f"_{grand_gagnant['nom'].replace(' ', '_').replace('.', '').lower()}"

evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', f'{file_suffix}.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', f'{file_suffix}.md'),
    temps_vlm=0.0
)

_ = compute_autopsy(
    sorted_idx_t2i_base, 
    sorted_idx_t2i_final, 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    mask_vlm_called=mask_vlm_called_list
)

'''

'\n\ndef evaluate_fusion(scores_dict, targets_gpu, method="additive", alpha=0.5, rrf_k=60, top_k=10, cascade_threshold=None):\n    """Calcule le R@1 en simulant dynamiquement le Top-K et la Cascade."""\n    correct_count = 0\n    total = len(scores_dict)\n    reranked_results = {}\n    mask_vlm_called = {} # Pour garder la trace des requêtes sauvées par la cascade\n\n    for q_idx_str, data in scores_dict.items():\n        q_idx = int(q_idx_str)\n        target = targets_gpu[q_idx].item()\n\n        ids = data["candidate_ids"][:top_k]\n        vlm = np.array(data["vlm_scores"][:top_k])\n        mirage = np.array(data["mirage_scores"][:top_k])\n\n        # 🛡️ --- LOGIQUE DE CASCADE --- 🛡️\n        ecart_confiance = mirage[0] - mirage[1] if len(mirage) > 1 else 0\n\n        if cascade_threshold is not None and ecart_confiance > cascade_threshold:\n            # MIRAGE est sûr de lui : on valide sans utiliser le VLM\n            new_top_k = ids\n            mask_vlm_called[q_idx] = False\

In [15]:

passes_extraction = [
    {
        "nom": "Validation",
        "limite": len(targets_t2i_val_gpu),
        "S_base": S_t2i_val_stage1,
        "dataset": val_dataset,
        "textes": val_texts,
        "fichier_out": os.path.join(config.RESULTS_DIR, "scores_bruts_t2i_val.json")
    },
    {
        "nom": "Test",
        "limite": limit_t2i,
        "S_base": S_t2i_stage1,
        "dataset": dataset,
        "textes": all_texts,
        "fichier_out": os.path.join(config.RESULTS_DIR, "scores_bruts_t2i_test.json")
    }
]

# 3. BOUCLE D'EXTRACTION EXHAUSTIVE
for passe in passes_extraction:
    print(f"\n" + "="*50)
    print(f"🚀 DÉMARRAGE EXTRACTION EXHAUSTIVE : Set de {passe['nom']}")
    print("="*50)
    
    fichier_out = passe['fichier_out']
    S_base = passe['S_base']
    limite = passe['limite']
    data_images = passe['dataset']
    data_textes = passe['textes']
    
    # Recalcul du Top 10 de base pour le set en cours
    sorted_idx_base = torch.argsort(S_base, dim=1, descending=True)
    scores_all_queries = {}

    # Chargement du cache existant
    if os.path.exists(fichier_out):
        with open(fichier_out, 'r') as f:
            scores_all_queries = json.load(f)

    for q_idx in tqdm(range(limite), desc=f"Extraction Pointwise ({passe['nom']})"):
        str_qidx = str(q_idx)
        top_10 = sorted_idx_base[q_idx, :10].tolist()
        
        if str_qidx in scores_all_queries:
            continue

        # 1. Récupération des images et de la requête
        images_pil = [data_images[idx]['image'] for idx in top_10]
        requete = data_textes[q_idx]
        prompt = f"Does the following image exactly and perfectly match this description: '{requete}'? Answer strictly with 'Yes' or 'No'."
        
        # 2. Inférence VLM (Scoring Pointwise par Batch)
        scores_vlm_p1 = vlm.score_image_pointwise_batch([prompt] * 5, images_pil[:5])
        scores_vlm_p2 = vlm.score_image_pointwise_batch([prompt] * 5, images_pil[5:])
        scores_vlm = scores_vlm_p1 + scores_vlm_p2
        
        # 3. Récupération des scores MIRAGE bruts
        scores_mirage = [S_base[q_idx, idx].item() for idx in top_10]
        
        # 4. Sauvegarde des données brutes
        scores_all_queries[str_qidx] = {
            "candidate_ids": top_10,
            "vlm_scores": scores_vlm,     # Probabilités [0, 1]
            "mirage_scores": scores_mirage # Scores de similarité phase 1
        }
        

        with open(fichier_out, 'w') as f:
            json.dump(scores_all_queries, f)

    # Sauvegarde finale pour ce set
    with open(fichier_out, 'w') as f:
        json.dump(scores_all_queries, f)

    print(f"✅ Extraction totale terminée et sauvegardée dans : {fichier_out}")


🚀 DÉMARRAGE EXTRACTION EXHAUSTIVE : Set de Validation


Extraction Pointwise (Validation): 100%|██████████| 5070/5070 [00:00<00:00, 130257.09it/s]

✅ Extraction totale terminée et sauvegardée dans : ./data/flickr30k/results/scores_bruts_t2i_val.json

🚀 DÉMARRAGE EXTRACTION EXHAUSTIVE : Set de Test


Extraction Pointwise (Test): 100%|██████████| 5000/5000 [00:00<00:00, 131159.71it/s]


✅ Extraction totale terminée et sauvegardée dans : ./data/flickr30k/results/scores_bruts_t2i_test.json


In [16]:
def evaluate_fusion(scores_dict, targets_gpu, method="additive", alpha=0.5, rrf_k=60, top_k=10, cascade_threshold=None):
    """Calcule le R@1 en simulant dynamiquement le Top-K et la Cascade."""
    correct_count = 0
    total = len(scores_dict)
    reranked_results = {}
    mask_vlm_called = {} # Pour garder la trace des requêtes sauvées par la cascade
    
    for q_idx_str, data in scores_dict.items():
        q_idx = int(q_idx_str)
        target = targets_gpu[q_idx].item()
        
        ids = data["candidate_ids"][:top_k]
        vlm = np.array(data["vlm_scores"][:top_k])
        mirage = np.array(data["mirage_scores"][:top_k])
        
        # 🛡️ --- LOGIQUE DE CASCADE --- 🛡️
        ecart_confiance = mirage[0] - mirage[1] if len(mirage) > 1 else 0
        
        if cascade_threshold is not None and ecart_confiance > cascade_threshold:
            # MIRAGE est sûr de lui : on valide sans utiliser le VLM
            new_top_k = ids
            mask_vlm_called[q_idx] = False
        else:
            # MIRAGE hésite : on fait la fusion
            mask_vlm_called[q_idx] = True
            mirage_norm = (mirage - mirage.min()) / (mirage.max() - mirage.min() + 1e-8)
            
            if method == "additive":
                final_scores = (alpha * vlm) + ((1 - alpha) * mirage_norm)
            elif method == "multiplicative":
                final_scores = vlm * mirage_norm
            elif method == "maximum":
                final_scores = np.maximum(vlm, mirage_norm)
            elif method == "concatenation":
                final_scores = (vlm * 1000.0) + mirage_norm
            elif method == "rrf":
                rank_vlm = np.argsort(np.argsort(-vlm)) + 1
                rank_mirage = np.argsort(np.argsort(-mirage_norm)) + 1
                final_scores = (1.0 / (rrf_k + rank_vlm)) + (1.0 / (rrf_k + rank_mirage))
            elif method == "vlm_seul":
                final_scores = vlm
            elif method == "harmonique":
                final_scores = 2 * (vlm * mirage_norm) / (vlm + mirage_norm + 1e-8)
            elif method == "geometrique":
                final_scores = np.sqrt(vlm * mirage_norm)
            elif method == "rank_penalty":
                rank_mirage = np.argsort(np.argsort(-mirage))
                final_scores = vlm - (0.05 * rank_mirage)
            elif method == "tie_breaker_mirage":
                final_scores = vlm + (0.05 * mirage_norm)
            elif method == "safe_vlm":
                rank_mirage = np.argsort(np.argsort(-mirage))
                final_scores = vlm - (0.02 * rank_mirage)
            elif method == "mirage_trust_on_ties":
                # Le VLM reste le maître absolu
                final_scores = np.copy(vlm)
                
                # mirage_norm va de 1.0 (le meilleur selon MIRAGE) à 0.0 (le pire)
                # On lui donne un "poids de départage" de 0.01 maximum.
                # Conséquence : MIRAGE ne pourra inverser le classement VLM QUE SI 
                # la différence entre deux images VLM est inférieure à 0.01 !
                final_scores += (0.01 * mirage_norm)
                
            best_indices = np.argsort(-final_scores) 
            new_top_k = [ids[i] for i in best_indices]
            
        reranked_results[q_idx] = new_top_k
        if new_top_k[0] == target:
            correct_count += 1
            
    return correct_count / total, reranked_results, mask_vlm_called

# ==========================================
# 0. PRÉPARATION DES DONNÉES (VAL ET TEST)
# ==========================================
RAW_SCORES_VAL_FILE = os.path.join(config.RESULTS_DIR, "scores_bruts_t2i_val.json")
RAW_SCORES_TEST_FILE = os.path.join(config.RESULTS_DIR, "scores_bruts_t2i_test.json")

with open(RAW_SCORES_VAL_FILE, 'r') as f:
    scores_bruts_val = json.load(f)
with open(RAW_SCORES_TEST_FILE, 'r') as f:
    scores_bruts_test = json.load(f)

# On recalcule les classements de base pour les deux sets
sorted_idx_t2i_val_base = torch.argsort(S_t2i_val_stage1, dim=1, descending=True)
sorted_idx_t2i_test_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)

# ==========================================
# 1. PHASE 1 : GRID SEARCH SUR VALIDATION
# ==========================================
print("\n" + "="*70)
print("🧪 PHASE 1 : RECHERCHE DES HYPERPARAMÈTRES (SET DE VALIDATION)")
print("="*70)

options_top_k = [5, 10]
methodes_simples = ["vlm_seul", "multiplicative", "maximum", "concatenation", "rrf", 
                    "harmonique", "geometrique", "rank_penalty", "tie_breaker_mirage", "safe_vlm"]

print("🎯 Calibrage des seuils de cascade sur le set de validation...")
options_cascade = {"Sans Cascade": None}
recalls_a_tester = [0.99, 0.995, 0.999]

for tr in recalls_a_tester:
    seuil = calibrate_confidence_threshold(S_t2i_val_stage1, targets_t2i_val_gpu, is_i2t=False, target_recall=tr)
    options_cascade[f"Avec (Recall={tr:.2f})"] = seuil

resultats_val = []
grand_gagnant_val = {"r1": 0, "nom": "", "params": {}}

# Exécution de toutes les combinaisons sur le VAL
for top_k in options_top_k:
    for nom_cascade, val_cascade in options_cascade.items():
        
        # 1. Méthodes simples
        for methode in methodes_simples:
            r1, _, mask = evaluate_fusion(scores_bruts_val, targets_t2i_val_gpu, method=methode, top_k=top_k, cascade_threshold=val_cascade)
            req_sauvees = sum(not v for v in mask.values())
            
            resultats_val.append({
                "Profondeur": f"Top {top_k}",
                "Cascade": nom_cascade.split(" ")[0],
                "Stratégie de Fusion": methode.capitalize(),
                "R@1 Val (%)": r1 * 100,
                "VLM Évités": req_sauvees
            })
            
            if r1 > grand_gagnant_val["r1"]:
                grand_gagnant_val = {"r1": r1, "nom": f"{methode.capitalize()}_Top{top_k}_{nom_cascade.split(' ')[0]}", 
                                     "params": {"method": methode, "alpha": 0.5, "top_k": top_k, "cascade_threshold": val_cascade}}

        # 2. Additif (On scanne tout l'espace de 0 à 1 pour être sûr de trouver le vrai optimum global)
        best_add_r1, best_alpha = 0, 0
        best_add_mask = {}
        for a in np.linspace(0, 1, 21):
            r1, _, mask = evaluate_fusion(scores_bruts_val, targets_t2i_val_gpu, method="additive", alpha=a, top_k=top_k, cascade_threshold=val_cascade)
            if r1 > best_add_r1:
                best_add_r1 = r1
                best_alpha = a
                best_add_mask = mask
                
        req_sauvees_add = sum(not v for v in best_add_mask.values())
        resultats_val.append({
            "Profondeur": f"Top {top_k}",
            "Cascade": nom_cascade.split(" ")[0],
            "Stratégie de Fusion": f"Grid Search (Alpha={best_alpha:.2f})",
            "R@1 Val (%)": best_add_r1 * 100,
            "VLM Évités": req_sauvees_add
        })
        
        if best_add_r1 > grand_gagnant_val["r1"]:
            grand_gagnant_val = {"r1": best_add_r1, "nom": f"Grid_Search_A{best_alpha:.2f}_Top{top_k}_{nom_cascade.split(' ')[0]}", 
                                 "params": {"method": "additive", "alpha": best_alpha, "top_k": top_k, "cascade_threshold": val_cascade}}

# Affichage du classement Validation
df_val = pd.DataFrame(resultats_val).sort_values(by=["R@1 Val (%)", "VLM Évités"], ascending=[False, False]).reset_index(drop=True)
display(df_val.head(10)) # On n'affiche que le Top 10 pour ne pas surcharger

print("\n" + "⭐ "*15)
print(f"LA MEILLEURE CONFIGURATION (VAL) EST : {grand_gagnant_val['nom']}")
print(f"R@1 sur Validation : {grand_gagnant_val['r1']*100:.2f} %")
print("⭐ "*15 + "\n")


# ==========================================
# 2. PHASE 2 : APPLICATION SUR LE TEST
# ==========================================
print("🔒 Hyperparamètres verrouillés. Déploiement sur le Set de Test...")
print("="*70)

# On extrait les paramètres parfaits
p = grand_gagnant_val["params"]

# 🎯 Un seul appel à la fonction sur le set de Test !
test_r1, dict_test, mask_test = evaluate_fusion(
    scores_bruts_test, 
    targets_t2i_gpu, 
    method=p["method"], 
    alpha=p["alpha"], 
    top_k=p["top_k"], 
    cascade_threshold=p["cascade_threshold"]
)

print(f"\n🏆 PERFORMANCE FINALE RÉELLE (R@1 TEST) : {test_r1*100:.2f} % 🏆")
print(f"⚡ VLM économisés sur le Test : {sum(not v for v in mask_test.values())} requêtes\n")

# ==========================================
# 3. AUTOPSIE ET SAUVEGARDE FINALE
# ==========================================
sorted_idx_t2i_final = sorted_idx_t2i_test_base.clone()

# Reconstruction
for q_idx_str, new_top_k in dict_test.items():
    q_idx = int(q_idx_str)
    ordre_final = new_top_k + sorted_idx_t2i_test_base[q_idx, len(new_top_k):].tolist()
    sorted_idx_t2i_final[q_idx, :len(ordre_final)] = torch.tensor(ordre_final, device=device)

# Masque pour autopsie
mask_vlm_called_list = [False] * limit_t2i
for q_idx_str, was_called in mask_test.items():
    mask_vlm_called_list[int(q_idx_str)] = was_called

file_suffix = f"_{grand_gagnant_val['nom'].replace(' ', '_').replace('.', '').lower()}_strict_test"

evaluate_and_save_results(
    sorted_idx_t2i_test_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', f'{file_suffix}.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', f'{file_suffix}.md'),
    temps_vlm=0.0
)

_ = compute_autopsy(
    sorted_idx_t2i_test_base, 
    sorted_idx_t2i_final, 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    mask_vlm_called=mask_vlm_called_list
)


🧪 PHASE 1 : RECHERCHE DES HYPERPARAMÈTRES (SET DE VALIDATION)
🎯 Calibrage des seuils de cascade sur le set de validation...
📊 Seuil calibré pour attraper 99.0% des erreurs de validation : 0.05141
📊 Seuil calibré pour attraper 99.5% des erreurs de validation : 0.05486
📊 Seuil calibré pour attraper 99.9% des erreurs de validation : 0.06283


,Profondeur,Cascade,Stratégie de Fusion,R@1 Val (%),VLM Évités
0,Top 10,Avec,Grid Search (Alpha=0.75),89.506903,2705
1,Top 10,Avec,Grid Search (Alpha=0.55),89.447732,2415
2,Top 10,Sans,Grid Search (Alpha=0.55),89.447732,0
3,Top 5,Avec,Grid Search (Alpha=0.80),89.349112,2705
4,Top 5,Avec,Grid Search (Alpha=0.80),89.289941,2415
5,Top 10,Avec,Multiplicative,89.250493,2705
6,Top 10,Avec,Geometrique,89.250493,2705
7,Top 10,Avec,Multiplicative,89.250493,2415
8,Top 10,Avec,Geometrique,89.250493,2415
9,Top 5,Sans,Grid Search (Alpha=0.75),89.211045,0



⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ 
LA MEILLEURE CONFIGURATION (VAL) EST : Grid_Search_A0.75_Top10_Avec
R@1 sur Validation : 89.51 %
⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ 

🔒 Hyperparamètres verrouillés. Déploiement sur le Set de Test...

🏆 PERFORMANCE FINALE RÉELLE (R@1 TEST) : 89.70 % 🏆
⚡ VLM économisés sur le Test : 2549 requêtes


COMPARAISON FINALE : LATE FUSION vs VLM (T2I)
                                         R@1   R@5  R@10   mAP  NDCG  Temps (s)
1. Avant (Stage 1 Fusion)              0.868 0.974 0.988 0.916 0.936      0.000
2. Après (MIRAGE + VLM (Cascade LoRA)) 0.897 0.981 0.988 0.934 0.950      0.000

✅ Résultats globaux (avec temps) sauvegardés dans resultats_t2i_grid_search_a075_top10_avec_strict_test.csv et .md

--- BILAN COMPTABLE T2I ---
Total requêtes analysées : 2451
✅ Sauvetages (Faux -> Vrai) : 232
⚠️ Sabotages (Vrai -> Faux) : 89
❌ Doubles Échecs (Faux -> Faux) : 420
🆗 Confirmations (Vrai -> Vrai) : 1710
📈 GAIN NET DE R@1 : 143 requêtes (soit +2.86%)


In [17]:
# ==========================================
# 🔍 ANALYSE D'ERREURS : L'ÉCART DES SCORES (T2I)
# ==========================================
import numpy as np

print("\n" + "="*70)
print("🔍 AUTOPSIE DES ERREURS : SONT-ELLES DE 'JUSTESSE' ?")
print("="*70)

erreurs_analysees = []

# On utilise les paramètres gagnants pour recalculer les scores finaux exacts
p = grand_gagnant_val["params"]
alpha = p["alpha"]
method = p["method"]
top_k = p["top_k"]

for raw_key, top_k_final in dict_test.items():
    q_idx = int(raw_key)
    q_idx_str = str(raw_key) # 🛡️ On force en texte pour le JSON !
    target = targets_t2i_gpu[q_idx].item()
    
    # 1. On ne regarde QUE les erreurs (Le n°1 n'est pas la target)
    prediction = top_k_final[0]
    if prediction != target:
        
        # 2. On vérifie si la bonne image était quand même dans le Top K analysé
        if target in top_k_final:
            
            # 3. Récupération des données brutes avec la clé en string
            data = scores_bruts_test[q_idx_str]
            ids = data["candidate_ids"][:top_k]
            vlm = np.array(data["vlm_scores"][:top_k])
            mirage = np.array(data["mirage_scores"][:top_k])
            
            # 4. Recalcul des scores fusionnés (comme dans evaluate_fusion)
            mirage_norm = (mirage - mirage.min()) / (mirage.max() - mirage.min() + 1e-8)
            
            if method == "additive":
                final_scores = (alpha * vlm) + ((1 - alpha) * mirage_norm)
            elif method == "multiplicative":
                final_scores = vlm * mirage_norm
            elif method == "maximum":
                final_scores = np.maximum(vlm, mirage_norm)
            elif method == "vlm_seul":
                final_scores = vlm
            elif method == "smart_tie_breaker":
                final_scores = np.copy(vlm)
                final_scores = final_scores + (0.02 * mirage_norm)
            elif method == "mirage_bonus_top1":
                final_scores = np.copy(vlm)
                final_scores[0] += 0.05
            
            # 5. Extraction des scores qui nous intéressent
            idx_prediction = ids.index(prediction)
            idx_target = ids.index(target)
            
            score_pred = final_scores[idx_prediction]
            score_target = final_scores[idx_target]
            
            ecart = score_pred - score_target
            
            erreurs_analysees.append({
                "q_idx": q_idx,
                "score_erreur": score_pred,
                "score_verite": score_target,
                "ecart": ecart,
                "position_verite": top_k_final.index(target) + 1 # 2ème, 3ème...
            })

# Tri des erreurs par écart de justesse (les plus proches en premier)
erreurs_analysees = sorted(erreurs_analysees, key=lambda x: x["ecart"])

# Affichage des 10 erreurs les plus frustrantes
print(f"Nombre total d'erreurs analysables (Target dans le Top {top_k}) : {len(erreurs_analysees)}")
print("\nVoici les 10 erreurs qui se sont jouées à la plus petite marge :")
print("-" * 60)

for err in erreurs_analysees[:30]:
    print(f"Requête n°{err['q_idx']:<4} | "
          f"Score Erreur : {err['score_erreur']:.4f} | "
          f"Score Vérité : {err['score_verite']:.4f} | "
          f"ÉCART : {err['ecart']:.5f} (Target finie {err['position_verite']}ème)")

print("-" * 60)


🔍 AUTOPSIE DES ERREURS : SONT-ELLES DE 'JUSTESSE' ?
Nombre total d'erreurs analysables (Target dans le Top 10) : 456

Voici les 10 erreurs qui se sont jouées à la plus petite marge :
------------------------------------------------------------
Requête n°4995 | Score Erreur : 0.8486 | Score Vérité : 0.8603 | ÉCART : -0.01169 (Target finie 3ème)
Requête n°4004 | Score Erreur : 0.9720 | Score Vérité : 0.9712 | ÉCART : 0.00082 (Target finie 2ème)
Requête n°57   | Score Erreur : 0.9298 | Score Vérité : 0.9285 | ÉCART : 0.00131 (Target finie 2ème)
Requête n°3322 | Score Erreur : 0.9865 | Score Vérité : 0.9849 | ÉCART : 0.00159 (Target finie 2ème)
Requête n°4778 | Score Erreur : 0.6732 | Score Vérité : 0.6716 | ÉCART : 0.00160 (Target finie 2ème)
Requête n°2984 | Score Erreur : 0.9549 | Score Vérité : 0.9526 | ÉCART : 0.00238 (Target finie 2ème)
Requête n°2204 | Score Erreur : 0.9853 | Score Vérité : 0.9828 | ÉCART : 0.00254 (Target finie 2ème)
Requête n°3194 | Score Erreur : 0.9549 | Score 

In [18]:
# ==========================================
# 📊 ANALYSE DÉTAILLÉE DU RECALL (R@1 à R@5)
# ==========================================
print("\n" + "="*50)
print("📊 DISTRIBUTION DES PERFORMANCES (R@1 à R@5) T2I")
print("="*50)

recalls_t2i = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
total_queries_t2i = len(dict_test)

for q_idx_str, new_top_k in dict_test.items():
    q_idx = int(q_idx_str)
    
    # Récupération de la target (T2I = une seule image cible)
    target = targets_t2i_gpu[q_idx].item()
        
    # Calcul des Hits pour chaque profondeur (de 1 à 5)
    for k in range(1, 6):
        if target in new_top_k[:k]:
            recalls_t2i[k] += 1

# Affichage des résultats
for k in range(1, 6):
    score = (recalls_t2i[k] / total_queries_t2i) * 100
    print(f"R@{k} : {score:.2f} %")
print("="*50 + "\n")


📊 DISTRIBUTION DES PERFORMANCES (R@1 à R@5) T2I
R@1 : 89.70 %
R@2 : 94.84 %
R@3 : 96.78 %
R@4 : 97.68 %
R@5 : 98.10 %



In [19]:
# %%
# ======================================================================
# ⚔️ PHASE 4 : DUEL CIBLÉ (CO-T UNIQUEMENT SUR LES CAS INCERTAINS)
# ======================================================================
import numpy as np
import os
import json
from tqdm.auto import tqdm
from qwen_vl_utils import process_vision_info

print("\n" + "="*70)
print("⚔️ PHASE 4 : DUEL 'SUDDEN DEATH' CIBLÉ (Routage par Incertitude)")
print("="*70)

SEUIL_INCERTITUDE = 0.01 # Si l'écart entre le 1er et le 2ème est inférieur à 1%, on lance le Duel

# Fichier de cache du Sudden Death précédent (on ne jette rien !)
SUDDEN_DEATH_T2I_FILE = os.path.join(config.RESULTS_DIR, "scores_sudden_death_cot_t2i.json")

if os.path.exists(SUDDEN_DEATH_T2I_FILE):
    with open(SUDDEN_DEATH_T2I_FILE, 'r') as f:
        sudden_death_results = json.load(f)
else:
    sudden_death_results = {}

# Récupération des paramètres gagnants de la Phase 3
p = grand_gagnant_val["params"]
alpha = p["alpha"]
method = p["method"]
top_k = p["top_k"]

dict_test_final_cible = {}
correct_count_sd_cible = 0
total_sd = len(dict_test)

duels_joues = 0
sauvetages_cibles = 0
sabotages_cibles = 0

for raw_key, top_k_final in tqdm(dict_test.items(), desc="Évaluation et Duels Ciblés"):
    q_idx = int(raw_key)
    q_idx_str = str(raw_key)
    target = targets_t2i_gpu[q_idx].item()
    
    # 1. Recalcul des scores exacts pour trouver l'écart
    data = scores_bruts_test[q_idx_str]
    ids = data["candidate_ids"][:top_k]
    vlm_scores = np.array(data["vlm_scores"][:top_k])
    mirage_scores = np.array(data["mirage_scores"][:top_k])
    
    mirage_norm = (mirage_scores - mirage_scores.min()) / (mirage_scores.max() - mirage_scores.min() + 1e-8)
    
    if method == "additive":
        final_scores = (alpha * vlm_scores) + ((1 - alpha) * mirage_norm)
    elif method == "multiplicative":
        final_scores = vlm_scores * mirage_norm
    elif method == "maximum":
        final_scores = np.maximum(vlm_scores, mirage_norm)
    elif method == "vlm_seul":
        final_scores = vlm_scores
        
    # On trie les scores du plus grand au plus petit
    sorted_indices = np.argsort(-final_scores)
    score_1er = final_scores[sorted_indices[0]]
    score_2eme = final_scores[sorted_indices[1]]
    
    ecart = score_1er - score_2eme
    
    nouveau_top_k = list(top_k_final)
    id_A = top_k_final[0]
    id_B = top_k_final[1]
    
    # 2. ROUTAGE DYNAMIQUE : Est-ce qu'on est incertain ?
    if ecart <= SEUIL_INCERTITUDE:
        duels_joues += 1
        
        # Le modèle hésite, on lance le Duel CoT !
        gagnant = "A" # Valeur par défaut
        
        # Utilisation du cache si disponible
        cache_key = f"{q_idx_str}_{id_A}_{id_B}" # Clé unique sécurisée
        
        if cache_key in sudden_death_results:
            gagnant = sudden_death_results[cache_key]
        elif q_idx_str in sudden_death_results: # Rétrocompatibilité avec ton ancien cache
            gagnant = sudden_death_results[q_idx_str]
        else:
            # ---> INFERENCE VLM (S'il n'était pas dans le cache) <---
            requete_texte = all_texts[q_idx]
            image_A = dataset[id_A]['image']
            image_B = dataset[id_B]['image']
            
            prompt = f"""You are an expert visual detective. I will provide you with Image A and Image B.
            Your task is to determine which image perfectly matches this exact description: '{requete_texte}'

            Think step-by-step:
            1. Analyze Image A: Does it contain all elements of the description? Are there any contradictions?
            2. Analyze Image B: Does it contain all elements of the description? Are there any contradictions?
            3. Compare them: Which one is the undisputed perfect match?

            After your analysis, you MUST conclude your response with: "Winner: A" or "Winner: B"."""
            
            messages = [{"role": "user", "content": [
                {"type": "image", "image": image_A, "max_pixels": 262144},
                {"type": "image", "image": image_B, "max_pixels": 262144},
                {"type": "text", "text": prompt}
            ]}]
            
            text_prompt = vlm.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = vlm.processor(text=[text_prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(device)
            
            outputs = vlm.model.generate(**inputs, max_new_tokens=256)
            generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, outputs)]
            reponse = vlm.processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0].strip()
            
            import re
            match = re.search(r'Winner:\s*([AB])', reponse, re.IGNORECASE)
            if match and match.group(1).upper() == "B":
                gagnant = "B"
            elif "image b" in reponse.lower().split(".")[-1]: 
                gagnant = "B"
                
            # Sauvegarde dans le cache
            sudden_death_results[cache_key] = gagnant
            with open(SUDDEN_DEATH_T2I_FILE, 'w') as f:
                json.dump(sudden_death_results, f)
        
        # 3. Application du résultat du duel
        if gagnant == "B":
            nouveau_top_k[0] = id_B
            nouveau_top_k[1] = id_A
            
            # Autopsie ciblée
            if id_A == target: 
                sabotages_cibles += 1 # Aïe, A était bon et on a mis B
            elif id_B == target:
                sauvetages_cibles += 1 # Bingo, B était bon et on l'a mis 1er
                
    dict_test_final_cible[q_idx_str] = nouveau_top_k
    
    # Sécurité Mode Test
    if q_idx < limit_t2i:
        if nouveau_top_k[0] == target:
            correct_count_sd_cible += 1

r1_final_cible = (correct_count_sd_cible / limit_t2i) * 100

print("\n" + "🎯 "*15)
print(f"SCORE ULTIME (APRÈS DUEL CIBLÉ SUR INCERTITUDE) : {r1_final_cible:.2f} %")
print("🎯 "*15)

print("\n--- BILAN DU DUEL CIBLÉ (Seuil < 0.01) ---")
print(f"Duels joués : {duels_joues} requêtes (Le reste a été validé d'office)")
print(f"✅ Sauvetages (Grâce au CoT) : {sauvetages_cibles}")
print(f"⚠️ Sabotages (À cause du CoT) : {sabotages_cibles}")
gain_net = sauvetages_cibles - sabotages_cibles
print(f"📈 GAIN NET : {gain_net} requêtes")
print("="*70 + "\n")

# Sauvegarde finale
sorted_idx_t2i_final_sd_cible = sorted_idx_t2i_test_base.clone()
for raw_key, nouveau_top_k in dict_test_final_cible.items():
    q_idx = int(raw_key)
    if q_idx < limit_t2i:
        ordre_final = nouveau_top_k + sorted_idx_t2i_test_base[q_idx, len(nouveau_top_k):].tolist()
        sorted_idx_t2i_final_sd_cible[q_idx, :len(ordre_final)] = torch.tensor(ordre_final, device=device)

file_suffix = "_duel_cible_incertitude"
evaluate_and_save_results(
    sorted_idx_t2i_test_base[:limit_t2i], 
    sorted_idx_t2i_final_sd_cible[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', f'{file_suffix}.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', f'{file_suffix}.md'),
    temps_vlm=0.0
)


⚔️ PHASE 4 : DUEL 'SUDDEN DEATH' CIBLÉ (Routage par Incertitude)


Évaluation et Duels Ciblés: 100%|██████████| 5000/5000 [00:00<00:00, 78153.96it/s]


🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 
SCORE ULTIME (APRÈS DUEL CIBLÉ SUR INCERTITUDE) : 89.68 %
🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 

--- BILAN DU DUEL CIBLÉ (Seuil < 0.01) ---
Duels joués : 95 requêtes (Le reste a été validé d'office)
✅ Sauvetages (Grâce au CoT) : 8
⚠️ Sabotages (À cause du CoT) : 9
📈 GAIN NET : -1 requêtes




COMPARAISON FINALE : LATE FUSION vs VLM (T2I)
                                         R@1   R@5  R@10   mAP  NDCG  Temps (s)
1. Avant (Stage 1 Fusion)              0.868 0.974 0.988 0.916 0.936      0.000
2. Après (MIRAGE + VLM (Cascade LoRA)) 0.897 0.981 0.988 0.934 0.950      0.000

✅ Résultats globaux (avec temps) sauvegardés dans resultats_t2i_duel_cible_incertitude.csv et .md
